In [ ]:
from google.colab import userdata
from googleapiclient.discovery import build

API_KEY = userdata.get('YOUTUBE_API_KEY')
print("Key terbaca, panjang:", len(API_KEY))

youtube = build('youtube', 'v3', developerKey=API_KEY)
res = youtube.channels().list(
    part='snippet,statistics',
    forHandle='@YouTube'
).execute()

item = res['items'][0]
print(item['snippet']['title'], '-', item['statistics']['subscriberCount'], 'subscribers')

Key terbaca, panjang: 39
YouTube - 46500000 subscribers


In [ ]:
import pandas as pd

HANDLES = {
    "RCTI":     ("@OfficialRCTI", "MNC"),
    "MNCTV":    ("@MNCTVOfficialId", "MNC"),
    "GTV":      ("@officialgtvid", "MNC"),
    "iNews":    ("@OfficialiNews", "MNC"),
    "SCTV":     ("@sctv.suryacitratelevisi", "Kompetitor"),
    "Indosiar": ("@indosiar_ivm", "Kompetitor"),
    "Trans7":   ("@TRANS7Official", "Kompetitor"),
    "Trans TV": ("@TRANSTVofficial", "Kompetitor"),
}

rows = []
for nama, (handle, grup) in HANDLES.items():
    res = youtube.channels().list(
        part='snippet,statistics,contentDetails',
        forHandle=handle
    ).execute()
    if not res.get('items'):
        print("TIDAK DITEMUKAN:", nama, handle)
        continue
    it = res['items'][0]
    rows.append({
        'channel_id': it['id'],
        'nama': nama,
        'judul_di_youtube': it['snippet']['title'],
        'group': grup,
        'subscribers': it['statistics'].get('subscriberCount'),
        'total_views': it['statistics'].get('viewCount'),
        'video_count': it['statistics'].get('videoCount'),
        'uploads_playlist_id': it['contentDetails']['relatedPlaylists']['uploads'],
        'snapshot_date': pd.Timestamp.now().date(),
    })

df_channels = pd.DataFrame(rows)
df_channels

,channel_id,nama,judul_di_youtube,group,subscribers,total_views,video_count,uploads_playlist_id,snapshot_date
0,UCE9-bV_MCGgLnH7v4HSApDg,RCTI,Official RCTI,MNC,4870000,1787369710,32761,UUE9-bV_MCGgLnH7v4HSApDg,2026-10-24
1,UC34xi8JB1AKSotPUKHywbaA,MNCTV,MNCTV OFFICIAL,MNC,17600000,8368338346,102688,UU34xi8JB1AKSotPUKHywbaA,2026-10-24
2,UCZgAny8kv3i7n_DdvTQznAA,GTV,officialgtvid,MNC,3910000,1482827027,57849,UUZgAny8kv3i7n_DdvTQznAA,2026-10-24
3,UCoSkllfpgmFHtbVK835QaQg,iNews,Official iNews,MNC,10700000,7995394829,395204,UUoSkllfpgmFHtbVK835QaQg,2026-10-24
4,UC_vsErcsq56hOscPHkG-aVw,SCTV,Surya Citra Televisi (SCTV),Kompetitor,23000000,2889913230,26025,UU_vsErcsq56hOscPHkG-aVw,2026-10-24
5,UCYqOeAXJm8yV9sJ8Ud3cR7A,Indosiar,Indosiar,Kompetitor,35700000,8685665131,18567,UUYqOeAXJm8yV9sJ8Ud3cR7A,2026-10-24
6,UC24_Z2L-8Ki183AI9zJJzNQ,Trans7,TRANS7 OFFICIAL,Kompetitor,27900000,16760564499,117435,UU24_Z2L-8Ki183AI9zJJzNQ,2026-10-24
7,UCIohHXwCEKxWCLvAguJ-GjA,Trans TV,TRANS TV Official,Kompetitor,18700000,9294300703,172295,UUIohHXwCEKxWCLvAguJ-GjA,2026-10-24


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RAW = '/content/drive/MyDrive/mnc-analytics/raw'
os.makedirs(RAW, exist_ok=True)

df_channels.to_csv(f'{RAW}/yt_channels.csv', index=False)
print(pd.read_csv(f'{RAW}/yt_channels.csv').shape)

Mounted at /content/drive
(8, 9)


In [ ]:
CUTOFF = pd.Timestamp('2025-10-01', tz='UTC')

def ambil_daftar_video(playlist_id, cutoff=CUTOFF):
    hasil = []
    page_token = None
    panggilan = 0
    selesai = False
    while not selesai:
        res = youtube.playlistItems().list(
            part='contentDetails',
            playlistId=playlist_id,
            maxResults=50,
            pageToken=page_token
        ).execute()
        panggilan += 1
        for it in res['items']:
            tgl = pd.Timestamp(it['contentDetails']['videoPublishedAt'])
            if tgl < cutoff:
                selesai = True
                break
            hasil.append({
                'video_id': it['contentDetails']['videoId'],
                'published_at_utc': tgl,
            })
        page_token = res.get('nextPageToken')
        if not page_token:
            break
    return hasil, panggilan

pl_id = df_channels.loc[df_channels['nama'] == 'Indosiar', 'uploads_playlist_id'].iloc[0]
daftar, n_panggilan = ambil_daftar_video(pl_id)

print("Jumlah video sejak 1 Okt 2025:", len(daftar))
print("Panggilan API (= unit kuota):", n_panggilan)
print("Terbaru:", daftar[0]['published_at_utc'])
print("Terlama:", daftar[-1]['published_at_utc'])


Jumlah video sejak 1 Okt 2025: 7439
Panggilan API (= unit kuota): 149
Terbaru: 2026-10-23 15:49:37+00:00
Terlama: 2025-10-01 03:51:30+00:00


In [ ]:
import os

ringkasan = []
for _, r in df_channels.iterrows():
    nama_file = r['nama'].replace(' ', '_')
    path = f"{RAW}/ids_{nama_file}.csv"

    if os.path.exists(path):
        d = pd.read_csv(path)
        print("SKIP (sudah ada):", r['nama'], len(d))
        ringkasan.append({'nama': r['nama'], 'jumlah_video': len(d), 'panggilan': 0})
        continue

    daftar, n = ambil_daftar_video(r['uploads_playlist_id'])
    d = pd.DataFrame(daftar)
    d.insert(0, 'channel_id', r['channel_id'])
    d.to_csv(path, index=False)
    ringkasan.append({'nama': r['nama'], 'jumlah_video': len(d), 'panggilan': n})
    print(r['nama'], len(d), n)

df_ringkasan = pd.DataFrame(ringkasan)
print("TOTAL video:", df_ringkasan['jumlah_video'].sum())
print("TOTAL unit (daftar ID):", df_ringkasan['panggilan'].sum())
df_ringkasan

SKIP (sudah ada): RCTI 6654
SKIP (sudah ada): MNCTV 19529
SKIP (sudah ada): GTV 19516
SKIP (sudah ada): iNews 19971
SKIP (sudah ada): SCTV 8623
SKIP (sudah ada): Indosiar 7371
SKIP (sudah ada): Trans7 10370
SKIP (sudah ada): Trans TV 14269
TOTAL video: 106303
TOTAL unit (daftar ID): 0


,nama,jumlah_video,panggilan
0,RCTI,6654,0
1,MNCTV,19529,0
2,GTV,19516,0
3,iNews,19971,0
4,SCTV,8623,0
5,Indosiar,7371,0
6,Trans7,10370,0
7,Trans TV,14269,0


In [ ]:
from googleapiclient.errors import HttpError

def ambil_detail(video_ids, channel_id, grup):
    hasil = []
    scraped = pd.Timestamp.now(tz='UTC')
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        res = youtube.videos().list(
            part='snippet,statistics,contentDetails',
            id=','.join(batch)
        ).execute()
        for v in res['items']:
            s = v['snippet']
            st = v.get('statistics', {})
            cd = v['contentDetails']
            hasil.append({
                'video_id': v['id'],
                'channel_id': channel_id,
                'group': grup,
                'title': s.get('title'),
                'published_at': s.get('publishedAt'),
                'duration': cd.get('duration'),
                'views': st.get('viewCount'),
                'likes': st.get('likeCount'),
                'comments': st.get('commentCount'),
                'tags': '|'.join(s.get('tags', [])),
                'category_id': s.get('categoryId'),
                'live_broadcast': s.get('liveBroadcastContent'),
                'thumbnail_url': s.get('thumbnails', {}).get('medium', {}).get('url'),
                'scraped_at': scraped,
            })
    return hasil

for _, r in df_channels.iterrows():
    nama_file = r['nama'].replace(' ', '_')
    out = f"{RAW}/videos_{nama_file}.csv"
    if os.path.exists(out):
        print("SKIP (sudah ada):", r['nama'])
        continue
    ids = pd.read_csv(f"{RAW}/ids_{nama_file}.csv")['video_id'].tolist()
    try:
        data = ambil_detail(ids, r['channel_id'], r['group'])
    except HttpError as e:
        print("BERHENTI di", r['nama'], "->", e)
        break
    pd.DataFrame(data).to_csv(out, index=False)
    print(r['nama'], "| diminta:", len(ids), "| diterima:", len(data))


SKIP (sudah ada): RCTI
SKIP (sudah ada): MNCTV
SKIP (sudah ada): GTV
SKIP (sudah ada): iNews
SKIP (sudah ada): SCTV
SKIP (sudah ada): Indosiar
SKIP (sudah ada): Trans7
SKIP (sudah ada): Trans TV


In [ ]:
daftar_df = []
for _, r in df_channels.iterrows():
    nama_file = r['nama'].replace(' ', '_')
    d = pd.read_csv(f"{RAW}/videos_{nama_file}.csv")
    d.insert(1, 'channel_name', r['nama'])
    daftar_df.append(d)

df_videos = pd.concat(daftar_df, ignore_index=True)
df_videos.to_csv(f"{RAW}/yt_videos.csv", index=False)

print("Shape:", df_videos.shape)
print("video_id duplikat:", df_videos['video_id'].duplicated().sum())
print("\nJumlah per channel:")
print(df_videos['channel_name'].value_counts())
print("\nKosong per kolom:")
print(df_videos.isna().sum())
print("\nRentang tanggal (UTC):")
print(df_videos['published_at'].min(), "->", df_videos['published_at'].max())

Shape: (106303, 15)
video_id duplikat: 0

Jumlah per channel:
channel_name
iNews       19971
MNCTV       19529
GTV         19516
Trans TV    14269
Trans7      10370
SCTV         8623
Indosiar     7371
RCTI         6654
Name: count, dtype: int64

Kosong per kolom:
video_id             0
channel_name         0
channel_id           0
group                0
title                0
published_at         0
duration             9
views                0
likes                0
comments            32
tags              4095
category_id          0
live_broadcast       0
thumbnail_url        0
scraped_at           0
dtype: int64

Rentang tanggal (UTC):
2025-10-01T00:15:03Z -> 2026-10-20T07:13:18Z


In [ ]:
print(df_videos['live_broadcast'].value_counts())
print("Durasi P0D:", (df_videos['duration'] == 'P0D').sum())
print(df_videos[df_videos['duration'].isna()][['channel_name', 'title', 'published_at', 'live_broadcast', 'views']])

live_broadcast
none        106280
upcoming        19
live             4
Name: count, dtype: int64
Durasi P0D: 14
      channel_name                                            title  \
92146     Trans TV              KONTRAKAN TIGA PINTU Eps. 6 - NOBAR   
92147     Trans TV         KONTRAKAN TIGA PINTU Eps. 2 - CARI JODOH   
92148     Trans TV      KONTRAKAN TIGA PINTU Eps. 3 - JIDAN DISUNAT   
92149     Trans TV     KONTRAKAN TIGA PINTU Eps. 5 - PEKERJAAN BARU   
92150     Trans TV       KONTRAKAN TIGA PINTU Eps. 9 - UNDIAN SABUN   
92151     Trans TV      KONTRAKAN TIGA PINTU Eps. 7 - KALUNG IDAMAN   
92152     Trans TV       KONTRAKAN TIGA PINTU Eps. 4 - KUNCI HILANG   
92153     Trans TV      KONTRAKAN TIGA PINTU Eps. 10 - MOTOR BODONG   
92154     Trans TV  KONTRAKAN TIGA PINTU Eps. 8 - SERAGAM PENGAJIAN   

               published_at live_broadcast  views  
92146  2026-10-19T11:34:33Z       upcoming      0  
92147  2026-10-19T11:34:30Z       upcoming      0  
92148  2026-10-19T11

In [ ]:
print(df_videos[df_videos['duration'] == 'P0D'][['channel_name', 'title', 'live_broadcast', 'views']])

      channel_name                                              title  \
0             RCTI  🔴 LIVE STREAMING TUKANG OJEK PENGKOLAN | D | 2...   
7             RCTI  🔴 LIVE STREAMING TUKANG OJEK PENGKOLAN | G | 2...   
8             RCTI  🔴 LIVE STREAMING TUKANG OJEK PENGKOLAN | F | 2...   
9             RCTI  🔴 LIVE STREAMING PUTRI UNTUK PANGERAN PILIHAN ...   
6654         MNCTV  🔴RATAPAN ANAK PUNGUT | LIVE CAHAYA HIKMAH | 20...   
6658         MNCTV  🔴KI SELUT MELAWAN NYAI MUNING | LIVE RADEN KIA...   
6659         MNCTV  🔴PETAKA SUSUK SAMBER LILIN | LIVE KISAH MISTER...   
6660         MNCTV  🔴PENGANTIN ALAM GAIB | LIVE KISAH MISTERI | 20...   
6661         MNCTV  🔴KAMERA PENJEMPUT MAUT | LIVE KISAH MISTERI | ...   
6662         MNCTV  🔴KUTUKAN GADIS DURHAKA | LIVE KISAH MISTERI | ...   
6668         MNCTV  🔴 1 CINTA 2 WANITA | LIVE CAHAYA HIKMAH | 20 O...   
49329        iNews                    🔴LIVE STREAMING INEWS TV 24 JAM   
92034     Trans TV                     LIVE | NONTO

In [ ]:
live_judul = df_videos['title'].str.contains('LIVE|🔴', case=False, na=False)
print(df_videos[live_judul].groupby('channel_name').size())


channel_name
GTV         1182
Indosiar      48
MNCTV       3314
RCTI         179
SCTV          50
Trans TV      56
Trans7        91
iNews       2392
dtype: int64


In [ ]:
!pip install google-play-scraper

In [ ]:
from google_play_scraper import app, reviews, Sort
import pandas as pd

APPS = {
    "RCTI+":   "com.fta.rctitv",
    "Vision+": "com.zte.iptvclient.android.idmnc",
    "Vidio":   "com.vidio.android",
}

for nama, app_id in APPS.items():
    info = app(app_id, lang='id', country='id')
    print(nama, "->", info['title'], "| developer:", info['developer'],
          "| rating:", round(info['score'], 2), "| installs:", info['installs'],
          "| jumlah ulasan:", info['reviews'])

for nama, app_id in APPS.items():
    hasil, token = reviews(app_id, lang='id', country='id',
                           sort=Sort.NEWEST, count=200)
    d = pd.DataFrame(hasil)
    print(nama, d.shape, "|", d['at'].min(), "->", d['at'].max())

RCTI+ -> RCTI+ Superapp | developer: PT MNC DIGITAL INDONESIA | rating: 3.11 | installs: 10.000.000+ | jumlah ulasan: 98456
Vision+ -> Vision+: Live, Sports & Series | developer: MNC Digital | rating: 4.02 | installs: 10.000.000+ | jumlah ulasan: 87605
Vidio -> Vidio: Sports, Movies, Series | developer: PT Vidio Dot Com | rating: 4.4 | installs: 100.000.000+ | jumlah ulasan: 280642
RCTI+ (200, 11) | 2026-07-28 09:06:54 -> 2026-10-22 12:15:16
Vision+ (200, 11) | 2026-07-27 14:51:35 -> 2026-10-23 02:14:51
Vidio (200, 11) | 2026-10-17 12:25:11 -> 2026-10-23 00:57:01


In [ ]:
import time
import pandas as pd
from google_play_scraper import reviews, Sort

CUTOFF = pd.Timestamp('2024-01-01')
MAX_ULASAN = 20000

APPS = {
    "RCTIplus":  "com.fta.rctitv",
    "Visionplus": "com.zte.iptvclient.android.idmnc",
    "Vidio":     "com.vidio.android",
}

def scrape_app(nama, app_id):
    semua, token, n_batch = [], None, 0
    while True:
        hasil, token = reviews(app_id, lang='id', country='id',
                               sort=Sort.NEWEST, count=200,
                               continuation_token=token)
        if not hasil:
            break
        semua.extend(hasil)
        n_batch += 1
        terlama = pd.Timestamp(hasil[-1]['at'])
        if n_batch % 5 == 0:
            print(nama, len(semua), "ulasan | terlama:", terlama)
        if n_batch % 10 == 0:
            pd.DataFrame(semua).to_csv(f"{RAW}/reviews_{nama}_partial.csv", index=False)
        if terlama < CUTOFF or len(semua) >= MAX_ULASAN:
            break
        time.sleep(2)

    d = pd.DataFrame(semua)
    d = d[d['at'] >= CUTOFF].drop(columns=['userName', 'userImage'], errors='ignore')
    d['app'] = nama
    d['scraped_at'] = pd.Timestamp.now()
    d.to_csv(f"{RAW}/reviews_{nama}.csv", index=False)
    print("SELESAI", nama, d.shape, "|", d['at'].min(), "->", d['at'].max())
    return d

for nama, app_id in APPS.items():
    if os.path.exists(f"{RAW}/reviews_{nama}.csv"):
        print("SKIP (sudah ada):", nama)
        continue
    scrape_app(nama, app_id)

SKIP (sudah ada): RCTIplus
SKIP (sudah ada): Visionplus
SKIP (sudah ada): Vidio


In [ ]:
daftar = []
for nama in ["RCTIplus", "Visionplus", "Vidio"]:
    daftar.append(pd.read_csv(f"{RAW}/reviews_{nama}.csv", parse_dates=['at']))
df_rev = pd.concat(daftar, ignore_index=True)
df_rev.to_csv(f"{RAW}/app_reviews.csv", index=False)

print("Shape:", df_rev.shape)
print("reviewId duplikat:", df_rev['reviewId'].duplicated().sum())
print("\nKosong per kolom:")
print(df_rev.isna().sum())

print("\nDistribusi skor (% per aplikasi):")
print(pd.crosstab(df_rev['app'], df_rev['score'], normalize='index').round(3) * 100)

df_rev['tanggal'] = df_rev['at'].dt.date
hari = df_rev.groupby(['app', 'tanggal']).size().sort_values(ascending=False)
print("\n5 hari teramai per aplikasi:")
print(hari.groupby(level='app').head(5))

Shape: (46649, 11)
reviewId duplikat: 0

Kosong per kolom:
reviewId                    0
content                     0
score                       0
thumbsUpCount               0
reviewCreatedVersion    12437
at                          0
replyContent            12265
repliedAt               12265
appVersion              12437
app                         0
scraped_at                  0
dtype: int64

Distribusi skor (% per aplikasi):
score          1    2    3    4     5
app                                  
RCTIplus    55.7  4.7  3.6  3.0  32.9
Vidio       40.7  7.1  6.9  7.2  38.2
Visionplus  66.8  4.6  2.8  1.8  24.1

5 hari teramai per aplikasi:
app         tanggal   
Visionplus  2024-04-29    2183
            2024-03-21    1201
RCTIplus    2024-04-29     716
Visionplus  2024-07-14     372
Vidio       2024-12-06     371
RCTIplus    2025-10-08     340
            2024-04-28     336
Visionplus  2024-01-14     314
Vidio       2025-08-11     311
RCTIplus    2024-11-15     284
          

In [ ]:
from google_play_scraper import app

meta = []
for nama, app_id in APPS.items():
    info = app(app_id, lang='id', country='id')
    h = info['histogram']
    meta.append({
        'app': nama, 'app_id': app_id,
        'title': info['title'], 'developer': info['developer'],
        'rating': info['score'], 'ratings_count': info['ratings'],
        'reviews_count': info['reviews'], 'installs': info['installs'],
        'rating_1': h[0], 'rating_2': h[1], 'rating_3': h[2],
        'rating_4': h[3], 'rating_5': h[4],
        'snapshot_date': pd.Timestamp.now().date(),
    })

df_meta = pd.DataFrame(meta)
df_meta.to_csv(f"{RAW}/app_meta.csv", index=False)
df_meta

,app,app_id,title,developer,rating,ratings_count,reviews_count,installs,rating_1,rating_2,rating_3,rating_4,rating_5,snapshot_date
0,RCTIplus,com.fta.rctitv,RCTI+ Superapp,PT MNC DIGITAL INDONESIA,3.112778,174094,98456,10.000.000+,70945,6974,7332,9176,79662,2026-10-24
1,Visionplus,com.zte.iptvclient.android.idmnc,"Vision+: Live, Sports & Series",MNC Digital,4.016426,165105,87605,10.000.000+,33696,3611,3985,8794,115014,2026-10-24
2,Vidio,com.vidio.android,"Vidio: Sports, Movies, Series",PT Vidio Dot Com,4.399394,799129,280642,100.000.000+,83481,13726,20929,62992,617996,2026-10-24


In [ ]:
pd.set_option('display.max_colwidth', 120)
df_rev['tanggal'] = pd.to_datetime(df_rev['at']).dt.date.astype(str)

# Versi: apakah dua kolom sama, dan kosongnya di mana
ada = df_rev.dropna(subset=['reviewCreatedVersion', 'appVersion'])
print("Baris dengan dua versi berbeda:", (ada['reviewCreatedVersion'] != ada['appVersion']).sum())
print("Proporsi versi kosong per aplikasi:")
print(df_rev.groupby('app')['appVersion'].apply(lambda s: s.isna().mean()).round(3))

# Balasan
df_rev['replied'] = df_rev['replyContent'].notna()
print("\nTingkat balasan per aplikasi:")
print(df_rev.groupby('app')['replied'].mean().round(3))
print("\n3 balasan paling sering:")
print(df_rev['replyContent'].value_counts().head(3))

# Lonjakan
spike = ((df_rev['app'].eq('Visionplus') & df_rev['tanggal'].isin(['2024-04-29', '2024-03-21'])) |
         (df_rev['app'].eq('RCTIplus') & df_rev['tanggal'].eq('2024-04-29')))
print("\nPersen bintang 1 di hari lonjakan:", round((df_rev[spike]['score'] == 1).mean() * 100, 1))
print("\nDistribusi skor TANPA hari lonjakan (%):")
print((pd.crosstab(df_rev.loc[~spike, 'app'], df_rev.loc[~spike, 'score'], normalize='index') * 100).round(1))
print("\nSampel isi ulasan di hari lonjakan:")
print(df_rev[spike].sample(10, random_state=1)[['app', 'score', 'content']])

Baris dengan dua versi berbeda: 0
Proporsi versi kosong per aplikasi:
app
RCTIplus      0.349
Vidio         0.259
Visionplus    0.199
Name: appVersion, dtype: float64

Tingkat balasan per aplikasi:
app
RCTIplus      0.482
Vidio         0.978
Visionplus    0.627
Name: replied, dtype: float64

3 balasan paling sering:
replyContent
Halo, Kak! Terima kasih untuk review dan bintangnya, kami sangat menghargainya. :) Kami berharap kamu bisa menikmati RCTI+ dengan baik dan lancar. Apabila kamu memerlukan bantuan kami atau saran untuk kami, silakan hubungi kami di cs@rctiplus.com. :)    759
Hi Kak...Terimakasih atas review-nya...Selamat menikmati tayangan Vision+.[MR]\n                                                                                                                                                                              588
Halo, Kak! Terima kasih untuk review dan bintangnya, kami sangat senang mendengar kamu menikmati aplikasi kami. :) Kami harap kamu selalu bisa menikmati 

CALL 1: SETUP


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, os
BASE = '/content/drive/MyDrive/mnc-analytics'
RAW = f'{BASE}/raw'
CLEAN = f'{BASE}/clean'
os.makedirs(CLEAN, exist_ok=True)

df = pd.read_csv(f'{RAW}/app_reviews.csv', parse_dates=['at', 'repliedAt', 'scraped_at'])
print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(46649, 11)


CALL 2: CLEANING


In [ ]:
# 1. Buang kolom kembar dan rapikan nama kolom
df = df.drop(columns=['reviewCreatedVersion'])
df = df.rename(columns={
    'reviewId': 'review_id', 'content': 'content_raw', 'thumbsUpCount': 'thumbs_up',
    'appVersion': 'app_version', 'replyContent': 'reply_content',
    'repliedAt': 'replied_at', 'at': 'review_dt'
})

# 2. UTC -> WIB (Colab berjalan dalam UTC)
def ke_wib(s):
    return s.dt.tz_localize('UTC').dt.tz_convert('Asia/Jakarta')

df['review_dt'] = ke_wib(df['review_dt'])
df['replied_at'] = ke_wib(df['replied_at'])

df['review_date'] = df['review_dt'].dt.date
df['month'] = df['review_dt'].dt.strftime('%Y-%m')
df['day_of_week'] = df['review_dt'].dt.day_name()
df['hour'] = df['review_dt'].dt.hour

# 3. Balasan developer
df['reply_content'] = df['reply_content'].str.strip()
df['replied'] = df['reply_content'].notna()
df['reply_hours'] = (df['replied_at'] - df['review_dt']).dt.total_seconds() / 3600
df['reply_is_template'] = df.groupby('reply_content')['review_id'].transform('count') >= 20

# 4. Versi mayor dan sentimen dasar
df['app_version_major'] = df['app_version'].str.extract(r'^(\d+\.\d+)')[0]
df['sentiment'] = np.select([df['score'] <= 2, df['score'] == 3],
                            ['Negatif', 'Netral'], default='Positif')

# 5. Flag hari lonjakan (berbasis aturan, bukan daftar tanggal manual)
harian = df.groupby(['app', 'review_date']).size().rename('n_harian').reset_index()
harian['median_app'] = harian.groupby('app')['n_harian'].transform('median')
harian['is_spike_day'] = (harian['n_harian'] >= 5 * harian['median_app']) & (harian['n_harian'] >= 100)
df = df.merge(harian[['app', 'review_date', 'is_spike_day']], on=['app', 'review_date'], how='left')

# 6. Simpan waktu sebagai WIB tanpa penanda zona (lebih aman untuk Sheets dan Looker Studio)
df['review_dt'] = df['review_dt'].dt.tz_localize(None)
df['replied_at'] = df['replied_at'].dt.tz_localize(None)

df.to_csv(f'{CLEAN}/app_reviews_clean_tahap1.csv', index=False)
print(df.shape)


(46649, 20)


CALL 3: CHECKS


In [ ]:
print("Hari lonjakan per aplikasi:")
print(df[df['is_spike_day']].groupby('app')['review_date'].nunique())
print(df[df['is_spike_day']].groupby(['app', 'review_date']).size().sort_values(ascending=False).head(12))

print("\nreply_hours negatif:", (df['reply_hours'] < 0).sum())
print("Median jam balasan per aplikasi:")
print(df.groupby('app')['reply_hours'].median().round(1))

kelompok = pd.cut(df['score'], [0, 2, 3, 5], labels=['Bintang 1-2', 'Bintang 3', 'Bintang 4-5'])
print("\nTingkat balasan per kelompok skor:")
print(df.groupby(['app', kelompok], observed=True)['replied'].mean().round(3).unstack())
print("\nProporsi balasan template:")
print(df[df['replied']].groupby('app')['reply_is_template'].mean().round(3))

print("\nProporsi versi kosong per tahun:")
print(df.groupby([df['review_dt'].dt.year, 'app'])['app_version'].apply(lambda s: s.isna().mean()).round(2).unstack())

Hari lonjakan per aplikasi:
app
RCTIplus      27
Vidio          8
Visionplus    14
Name: review_date, dtype: int64
app         review_date
Visionplus  2024-04-29     2159
            2024-03-21     1187
RCTIplus    2024-04-29      719
Vidio       2024-12-06      378
RCTIplus    2024-04-28      355
Visionplus  2024-07-14      353
            2024-01-14      314
RCTIplus    2024-11-15      283
Vidio       2025-08-11      281
RCTIplus    2024-05-09      275
            2024-09-10      272
Vidio       2025-08-12      272
dtype: int64

reply_hours negatif: 566
Median jam balasan per aplikasi:
app
RCTIplus      17.2
Vidio          1.5
Visionplus     0.6
Name: reply_hours, dtype: float64

Tingkat balasan per kelompok skor:
score       Bintang 1-2  Bintang 3  Bintang 4-5
app                                            
RCTIplus          0.411      0.639        0.584
Vidio             0.965      0.989        0.991
Visionplus        0.517      0.817        0.908

Proporsi balasan template:
app
RC

QUIDK DIAGNOSTIC TO RUN

In [ ]:
import pandas as pd

# A. timezone check: compare the raw file with the cleaned df
raw = pd.read_csv('/content/drive/MyDrive/mnc-analytics/raw/app_reviews.csv')  # adjust path if needed
print(raw.columns.tolist())
print(raw[['reviewId']].head(1))  # confirms the raw id column name
r = raw.set_index(raw.columns[0])  # assumes the first column is the review id
sample_ids = df[df.replied].review_id.head(3)
print(df.set_index('review_id').loc[sample_ids, ['review_dt', 'replied_at']])
print(r.loc[sample_ids].filter(like='at'))

# B. negative reply_hours: how many fall within 0 to -7h?
neg = df[df.reply_hours < 0]
print('neg within 0 to -7h:', ((neg.reply_hours > -7)).sum(), '/', len(neg))
print(df[df.reply_hours < -20000][['app', 'review_dt', 'replied_at', 'reply_hours']].head())
print('earliest replied_at:', df.replied_at.min())

# C. Oct 2025 spike check
d = pd.to_datetime(df.review_date)
print(df[(d >= '2025-10-06') & (d <= '2025-10-10')].groupby(['app', 'review_date']).size())

# D. spike threshold per app
daily = df.groupby(['app', 'review_date']).size().rename('n').reset_index()
print(daily.groupby('app').n.median())
print(daily[daily.n >= 100].sort_values(['app', 'review_date']).to_string())

['reviewId', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'app', 'scraped_at']
                               reviewId
0  4da3ff82-9d7d-468a-bbb9-30cab4c19a12
                                               review_dt          replied_at
review_id                                                                   
4da3ff82-9d7d-468a-bbb9-30cab4c19a12 2026-10-19 09:04:17 2026-10-19 09:57:48
f2e9b91a-6a48-4021-ade8-6ad72d073458 2026-10-18 19:21:36 2026-10-19 09:58:17
5c99d202-9d0e-4099-b4ef-f7ff1611a22e 2026-10-18 19:21:33 2026-10-19 10:00:20
                                     reviewCreatedVersion  \
reviewId                                                    
4da3ff82-9d7d-468a-bbb9-30cab4c19a12               2.63.5   
f2e9b91a-6a48-4021-ade8-6ad72d073458                  NaN   
5c99d202-9d0e-4099-b4ef-f7ff1611a22e               2.64.1   

                                                       at  \
reviewId                

In [ ]:
import re
import numpy as np

# 1. invalid reply times -> NaN + flag
df['reply_time_invalid'] = df['replied'] & (df['reply_hours'] < 0)
df.loc[df['reply_time_invalid'], 'reply_hours'] = np.nan

# 2. template detection on normalized text
def norm_reply(s):
    if pd.isna(s):
        return np.nan
    s = s.lower()
    s = re.sub(r'\[[a-z]{1,4}\]', ' ', s)   # signature tags like [MR]
    return re.sub(r'\s+', ' ', s).strip()

df['reply_norm'] = df['reply_content'].map(norm_reply)
cnt = df.groupby(['app', 'reply_norm'])['reply_norm'].transform('size')
df['reply_is_template'] = df['replied'] & (cnt >= 20)
df = df.drop(columns='reply_norm')

# 3. spike events: consecutive flagged days = one event
d = df[['app', 'review_date', 'is_spike_day']].drop_duplicates().copy()
d['review_date'] = pd.to_datetime(d['review_date'])
d = d[d.is_spike_day].sort_values(['app', 'review_date'])
gap = d.groupby('app')['review_date'].diff().dt.days.ne(1)
d['spike_event_id'] = d['app'] + '_' + gap.groupby(d['app']).cumsum().astype(str)
d['review_date'] = d['review_date'].dt.date
df = df.merge(d[['app', 'review_date', 'spike_event_id']], on=['app', 'review_date'], how='left')

# checks
print(df.shape)   # expect (46649, 22)
print(df.groupby('app')['reply_time_invalid'].sum())
print(df[df.replied].groupby('app')['reply_is_template'].mean())
neg = df[df.score <= 2]
print(neg.groupby('app')['replied'].mean())                           # all days
print(neg[~neg.is_spike_day].groupby('app')['replied'].mean())        # excl. spikes
print(df[df.replied & ~df.reply_is_template].groupby('app')['reply_hours'].median())

df.to_csv('/content/drive/MyDrive/mnc-analytics/clean/app_reviews_clean_tahap1.csv', index=False)

(46649, 22)
app
RCTIplus      180
Vidio         238
Visionplus    148
Name: reply_time_invalid, dtype: int64
app
RCTIplus      0.872028
Vidio         0.451293
Visionplus    0.681792
Name: reply_is_template, dtype: float64
app
RCTIplus      0.411480
Vidio         0.964604
Visionplus    0.517054
Name: replied, dtype: float64
app
RCTIplus      0.574350
Vidio         0.965339
Visionplus    0.891304
Name: replied, dtype: float64
app
RCTIplus      17.291111
Vidio          1.600972
Visionplus     0.913333
Name: reply_hours, dtype: float64


In [ ]:
import pandas as pd

PATH = '/content/drive/MyDrive/mnc-analytics/raw/yt_videos.csv'
v = pd.read_csv(PATH)

print('shape:', v.shape)
print('\ncolumns:', v.columns.tolist())
print('\ndtypes:\n', v.dtypes)
print('\nhead(3):\n', v.head(3).T)          # transposed so every column is readable

# nulls and duplicates
print('\nnulls:\n', v.isna().sum()[lambda s: s > 0])
print('\nduplicate video_id:', v.iloc[:, 0].duplicated().sum())

# date format (needed for the WIB conversion)
date_cols = [c for c in v.columns if 'publish' in c.lower() or 'date' in c.lower()]
print('\ndate cols:', date_cols)
for c in date_cols:
    print(c, '->', v[c].head(3).tolist())

# duration format
dur_cols = [c for c in v.columns if 'dur' in c.lower()]
for c in dur_cols:
    print('\n', c, 'samples:', v[c].dropna().sample(5, random_state=1).tolist())
    print(c, 'P0D:', (v[c] == 'P0D').sum(), '| null:', v[c].isna().sum())

# channel column and the group label
ch_cols = [c for c in v.columns if 'channel' in c.lower() or c.lower() == 'group']
for c in ch_cols:
    print('\n', c, 'unique:', v[c].nunique(), v[c].unique()[:8])

# live-like titles (since liveBroadcastContent isn't saved)
title_col = [c for c in v.columns if 'title' in c.lower()][0]
live_pat = r'^\s*(?:LIVE|🔴)|\bLIVE\s*(?:STREAMING|NOW)\b|SIARAN LANGSUNG'
is_live = v[title_col].str.contains(live_pat, case=False, regex=True, na=False)
print('\nlive-like titles:', is_live.sum())
print(v.loc[is_live, title_col].sample(min(8, is_live.sum()), random_state=1).tolist())

# MNCTV vs GTV: duplicate titles across channels
print('\nduplicate titles overall:', v[title_col].duplicated().sum())


shape: (106303, 15)

columns: ['video_id', 'channel_name', 'channel_id', 'group', 'title', 'published_at', 'duration', 'views', 'likes', 'comments', 'tags', 'category_id', 'live_broadcast', 'thumbnail_url', 'scraped_at']

dtypes:
 video_id           object
channel_name       object
channel_id         object
group              object
title              object
published_at       object
duration           object
views               int64
likes               int64
comments          float64
tags               object
category_id         int64
live_broadcast     object
thumbnail_url      object
scraped_at         object
dtype: object

head(3):
                                                                               0  \
video_id                                                            4e3LfCgMQVU   
channel_name                                                               RCTI   
channel_id                                             UCE9-bV_MCGgLnH7v4HSApDg   
group                 

Cleaning Script (stage 1, youtube)

In [ ]:
import pandas as pd, numpy as np

RAW = '/content/drive/MyDrive/mnc-analytics/raw/yt_videos.csv'
OUT = '/content/drive/MyDrive/mnc-analytics/clean/yt_videos_clean_tahap1.csv'
SHORTS_MAX, CLIP_MAX = 180, 600          # seconds

v = pd.read_csv(RAW)
n0 = len(v)

# 1. time: UTC -> WIB
pub = pd.to_datetime(v['published_at'], utc=True)
scr = pd.to_datetime(v['scraped_at'], utc=True)
wib = pub.dt.tz_convert('Asia/Jakarta').dt.tz_localize(None)
v['published_wib'] = wib
v['date']          = wib.dt.date
v['month']         = wib.dt.strftime('%Y-%m')
v['day_of_week']   = wib.dt.day_name()
v['dow_num']       = wib.dt.dayofweek          # Mon=0, for sorting in Looker
v['hour']          = wib.dt.hour

# 2. ISO 8601 duration -> seconds
pat = r'^P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?)?$'
bad = v['duration'].notna() & ~v['duration'].str.match(pat, na=False)
print('unparseable durations:', bad.sum())
p = v['duration'].str.extract(pat).astype(float).fillna(0)
v['duration_sec'] = p[0]*86400 + p[1]*3600 + p[2]*60 + p[3]
v.loc[v['duration'].isna(), 'duration_sec'] = np.nan
v['duration_min'] = (v['duration_sec'] / 60).round(2)

# 3. live / upcoming flags
v['is_live_now'] = v['live_broadcast'].eq('live')
v['is_upcoming'] = v['live_broadcast'].eq('upcoming')
v['exclude_perf'] = v['is_live_now'] | v['is_upcoming']   # flagged, not deleted
v['live_title']  = v['title'].str.contains(r'🔴|\bLIVE\b', regex=True, na=False)  # case-sensitive on purpose

# 4. format
v['format'] = np.select(
    [v['is_upcoming'],
     v['is_live_now'] | v['live_title'],
     v['duration_sec'].isna(),
     v['duration_sec'] <= SHORTS_MAX,
     v['duration_sec'] <= CLIP_MAX],
    ['Upcoming', 'Live/Streaming', 'Unknown', 'Shorts', 'Clip'],
    default='Full')

# 5. engagement, views per day, data-quality flags
v['comments_missing'] = v['comments'].isna()
ok = (v['views'] > 0) & ~v['comments_missing']
v['engagement_rate'] = np.where(ok, (v['likes'] + v['comments']) / v['views'] * 100, np.nan)
age_days = ((scr - pub).dt.total_seconds() / 86400).clip(lower=1)
v['views_per_day'] = (v['views'] / age_days).round(2)
v['likes_zero_flag'] = (v['likes'] == 0) & (v['views'] >= 1000)
v['dup_title_in_channel'] = v.duplicated(['channel_id', 'title'], keep=False)

# 6. outlier flag: log10(views) IQR fence inside each channel x format
lv = np.log10(v['views'] + 1)
g = lv.groupby([v['channel_name'], v['format']])
q1 = g.transform(lambda s: s.quantile(.25))
q3 = g.transform(lambda s: s.quantile(.75))
v['is_outlier'] = lv > q3 + 3 * (q3 - q1)

# ---- checks ----
print('\nshape:', v.shape, '(expect', n0, 'rows x 33 cols)')
print('published_wib range:', v['published_wib'].min(), '->', v['published_wib'].max())
print('\nlive now / upcoming / exclude_perf:', v.is_live_now.sum(), v.is_upcoming.sum(), v.exclude_perf.sum(), '(expect 4 / 19 / 23)')
print('\nformat by group (%):')
print((pd.crosstab(v['group'], v['format'], normalize='index') * 100).round(1))
print('\nformat counts per channel:')
print(pd.crosstab(v['channel_name'], v['format']))
print('\nlive-titled videos under 180s:', (v.live_title & (v.duration_sec <= SHORTS_MAX)).sum())
print('\nlikes_zero_flag per channel:\n', v.groupby('channel_name')['likes_zero_flag'].sum())
print('\nduplicate titles in channel (% of rows):')
print((v.groupby('channel_name')['dup_title_in_channel'].mean() * 100).round(1))
print('\noutliers per channel:\n', v.groupby('channel_name')['is_outlier'].sum())
print('\nengagement_rate NaN:', v['engagement_rate'].isna().sum())
print('\ntotal views per channel (save for the Excel reconciliation):')
print(v.groupby('channel_name')['views'].sum())

v.to_csv(OUT, index=False)

unparseable durations: 0

shape: (106303, 34) (expect 106303 rows x 33 cols)
published_wib range: 2025-10-01 07:15:03 -> 2026-10-20 14:13:18

live now / upcoming / exclude_perf: 4 19 23 (expect 4 / 19 / 23)

format by group (%):
format      Clip  Full  Live/Streaming  Shorts  Upcoming
group                                                   
Kompetitor  28.1  29.6             0.3    41.9       0.0
MNC         30.0  44.8            10.6    14.6       0.0

format counts per channel:
format        Clip  Full  Live/Streaming  Shorts  Upcoming
channel_name                                              
GTV           6664  8641            1167    3044         0
Indosiar      1593   998              41    4739         0
MNCTV         6489  8738            3303     993         6
RCTI          1369  2731             129    2422         3
SCTV          1594  2291              13    4725         0
Trans TV      3328  4475               7    6449        10
Trans7        4921  4251              76   

In [ ]:
import pandas as pd
v = pd.read_csv('/content/drive/MyDrive/mnc-analytics/clean/yt_videos_clean_tahap1.csv')

# A. Trans TV duplicate titles
t = v[(v.channel_name == 'Trans TV') & v.dup_title_in_channel]
print('rows:', len(t), '| distinct titles:', t.title.nunique())
print(t.groupby('title').agg(n=('video_id', 'size'),
                             durations=('duration_sec', 'nunique'),
                             days=('date', 'nunique'))
       .sort_values('n', ascending=False).head(10))

# B. category_id per channel
print(pd.crosstab(v.channel_name, v.category_id))

# C. sample titles + most frequent non-date segments per channel
date_re = r'\d{1,2}\s+[A-Za-z]+\s+20\d\d|\d{1,2}/\d{1,2}'
for ch, g in v.groupby('channel_name'):
    print(f'\n=== {ch} ===')
    print(g.title.sample(10, random_state=2).tolist())
    segs = g.title.str.split('|').explode().str.strip().str.upper()
    segs = segs[~segs.str.contains(date_re, regex=True, na=False)]
    print(segs.value_counts().head(20))

rows: 5265 | distinct titles: 2389
                                                                       n  \
title                                                                      
DISKON MELIMPAH HINGGA 50% PAKAI ALLO PAYLATER                       113   
Pengalaman Staycation Lengkap dalam Satu Lokasi - TRANS HOTEL GROUP   97   
JOGJA FINANCIAL FESTIVAL 2026                                         59   
JOGJA RUN D-CITY                                                      52   
TRANS HOTEL JAKARTA                                                   50   
TRANS LUXURY HOTEL SURABAYA                                           36   
26 September 2026                                                     22   
TRANS HOTEL GROUP                                                     14   
#TransTVPadelTournament                                               13   
TRANS HOTEL SURABAYA                                                  13   

                                                    

Stage 2 script (content_genre)


In [ ]:
import pandas as pd, numpy as np

IN  = '/content/drive/MyDrive/mnc-analytics/clean/yt_videos_clean_tahap1.csv'
OUT = '/content/drive/MyDrive/mnc-analytics/clean/yt_videos_clean_tahap2.csv'
SAMPLE = '/content/drive/MyDrive/mnc-analytics/final/genre_validation_sample.csv'
v = pd.read_csv(IN)
t = v['title'].fillna('').str.upper().str.replace(r'\s+', ' ', regex=True)

TRANS = ['Trans7', 'Trans TV']
RULES = [  # (genre, regex, channels to skip) -- first match wins
 ('Promo & Korporat', r'ALLO PAYLATER|TRANS (LUXURY )?HOTEL|JOGJA FINANCIAL|JOGJA RUN|PADEL|GRAND RECRUITMENT|TRANS7 MENGAJAR|SELAMAT BERBUKA|SAATNYA BERBUKA', None),
 ('Berita', r'INEWS|BULETIN|BREAKING NEWS|BERITA|HEADLINE|RAKYAT BERSUARA|SIKAT TIPIS|DEEP TALK|VIRAL TAPI HOAKS|\bVTH\b|INTERUPSI|BIKIN TERANG|IBREAK', None),
 ('Olahraga', r'\bLIGA (?!BINTANG)|SUPER LEAGUE|PIALA|TIMNAS|FIFA|ASEAN (CUP|CHAMPIONSHIP)|PERSIB|PERSIJA|AREMA|PERSEBAYA|MOTOGP|BADMINTON|HASIL PERTANDINGAN|KICK OFF', None),
 ('Olahraga', r'\bVS\b', ['iNews']),
 ('Religi', r'CAHAYA HATI|KAJIAN|TAUSIYAH|GEMA RAMADAN', None),
 ('Talent Show & Musik', r"D['’]? ?ACADEMY|\bDA ?\d|IDA 2026|BAND ACADEMY|THE ICON|SPEKTAKULER|KONSER|AKSI INDOSIAR|AKSI 2026|BEFORE ON AIR|\bIDOL\b|AUDITION|KILAU RAYA|LIGA BINTANG JUARA", None),
 ('Infotainment', r'GO ?SPOT|HOT ?SHOT|SILET|HALO SELEBRITI|INFOTAINMENT|BESTKISS', None),
 ('Variety & Lifestyle', r'UANG KAGET|BEDAH RUMAH|NEW FAMILY 100|DEG ?DEG-?AN|RUMAH TEKA TEKI|CAKEP CAKEP BAPER|1001 CARA MENGGAET|CEWEK CEWEK BADUNG|TAKE ME OUT|PAGI ?PAGI ?AMBYAR|BROWNIS|RUMPI|MONITOR KETUA|MAKAN RECEH|\bFYP\b|LAPOR PAK|PAS BUKA|JEJAK PETUALANG|TIBA[- ]TIBA KUIS|NGOBROL BARENG IDOLA|MANCUR|BIG IS BEAUTIFUL|COBOY JUNIOR|KARNAVAL|AWARDS', None),
 ('Sinetron & FTV', r'\bFTV\b|KUASA IL+AHI|KEMBALINYA RADEN KIAN SANTANG|FATIH DI KAMPUNG JAWARA|CAHAYA HIKMAH|CAHAYA MENTARI|SULTAN AJI|TUTUR TINULAR|SUPARMAN REBORN|NYI RORO KIDUL|DZOLIM|MISTERI GUNUNG MERAPI|KISAH MISTERI|IPA ?& ?IPS|OBSESI|ARJUNA|RAJA DAN AKU|KINARA|DIBALIK RAHASIA|DUNIA KIARA|RAHASIA BATIN|MENCINTAI IPAR|TERBELENGGU RINDU|TERIKAT JANJI|KAU DITAKD[IR]+KAN UNTUKKU|DUSTA DIBALIK CINTA|\bAIRA\b|99 NAMA CINTA|CINTA SEPENUH JIWA|PUTRI UNTUK PANGERAN|TURUN RANJANG|\bKDU\b|MERANGKAI KISAH INDAH|KISAH NYATA SPESIAL|MUTIARA HATI|MENGETUK PINTU HATI|ASMARA GEN ?Z|BERI CINTA WAKTU|CINTA SELUAS SAMUDRA|PUTRI PEWARIS|ISTRI SATU-?SATUNYA|PESUGIHAN|SILUMAN|TUKANG OJEK PENGKOLAN', None),
 ('Sinetron & FTV', r'\bEP(?:S|ISODE)?\.? ?#?\d|#EPS ?\d|\bPART\b|\(\d/\d\)', TRANS),  # generic episode markers
]

genre = pd.Series('Lainnya', index=v.index)
rule  = pd.Series('', index=v.index)
done  = pd.Series(False, index=v.index)
for i, (g, pat, skip) in enumerate(RULES):
    m = t.str.contains(pat, regex=True) & ~done
    if skip:
        m &= ~v['channel_name'].isin(skip)
    genre[m], rule[m], done[m] = g, f'{i}:{g}', True

m = (v['channel_name'] == 'iNews') & (v['category_id'] == 25) & ~done   # iNews default
genre[m], rule[m] = 'Berita', 'default_inews'

v['content_genre'], v['genre_rule'] = genre, rule
v['is_promo'] = v['content_genre'].eq('Promo & Korporat')

# ---- checks ----
print('shape:', v.shape)
print('\ngenre share per channel (%):')
print((pd.crosstab(v['channel_name'], v['content_genre'], normalize='index') * 100).round(1))
print('\ngenre x format (counts):')
print(pd.crosstab(v['content_genre'], v['format']))
un = v[v['content_genre'] == 'Lainnya']
print('\nLainnya share per channel (%):')
print((v['content_genre'].eq('Lainnya').groupby(v['channel_name']).mean() * 100).round(1))
segs = un['title'].str.upper().str.split('|').explode().str.strip()
print('\ntop unmatched segments:')
print(segs[~segs.str.contains(r'\d{1,2}[/ ]\d|#SHORTS', regex=True, na=False)].value_counts().head(40))
for ch, g in un.groupby('channel_name'):
    print(f'\n--- unmatched sample: {ch} ---')
    print(g['title'].sample(min(6, len(g)), random_state=3).tolist())

# 100-row random sample for manual accuracy check (label in the genre_manual column)
s = v.sample(100, random_state=42)[['video_id', 'channel_name', 'title', 'content_genre']].copy()
s['genre_manual'] = ''
s.to_csv(SAMPLE, index=False)
v.to_csv(OUT, index=False)

/tmp/ipykernel_3003/2950513654.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = t.str.contains(pat, regex=True) & ~done
/tmp/ipykernel_3003/2950513654.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = t.str.contains(pat, regex=True) & ~done


shape: (106303, 37)

genre share per channel (%):
content_genre  Berita  Infotainment  Lainnya  Olahraga  Promo & Korporat  \
channel_name                                                               
GTV               2.0           0.0     20.6       1.0               0.0   
Indosiar          0.2           0.1     36.2      19.7               0.0   
MNCTV             0.1           0.0     15.3       1.3               0.0   
RCTI              0.4           4.6     34.1       3.3               0.1   
SCTV              0.1           1.0     48.1       3.1               0.3   
Trans TV          0.2           0.0     32.9       0.5               3.7   
Trans7            0.1           0.0     54.5       1.3               1.9   
iNews            96.3           0.0      0.2       1.7               0.1   

content_genre  Religi  Sinetron & FTV  Talent Show & Musik  \
channel_name                                                 
GTV               0.0            46.1                  1.9   
Ind

In [ ]:
import re, unicodedata, warnings
warnings.filterwarnings('ignore', message='This pattern')

def clean(s):
    s = unicodedata.normalize('NFKC', s).upper()          # fancy bold letters -> plain
    s = re.sub(r'[\u200b-\u200f\u2060-\u2064\ufeff]', '', s)
    return re.sub(r'\s+', ' ', s).strip()

t = v['title'].fillna('').map(clean)
h = t.str.findall(r'#(\w+)').str.join(' ')                 # hashtags only, e.g. ASMARAGENZSCTV
TRANS = ['Trans7', 'Trans TV']
TALENT = 'Talent & Reality Show'

genre = pd.Series('Lainnya', index=v.index)
rule  = pd.Series('', index=v.index)
done  = pd.Series(False, index=v.index)

def put(mask, g, label):
    mask = mask & ~done
    genre[mask], rule[mask], done[mask] = g, label, True

has = lambda pat, src=t: src.str.contains(pat, regex=True)
ch  = v['channel_name']

put(has(r'ALLO PAYLATER|TRANS (?:LUXURY )?HOTEL|JOGJA FINANCIAL|JOGJA RUN|PADEL|GRAND RECRUITMENT|TRANS7 MENGAJAR|SELAMAT BERBUKA|SAATNYA BERBUKA|UPACARA PENURUNAN BENDERA'), 'Promo & Korporat', 'promo')
put((ch == 'iNews') & ((v['category_id'] == 17) | has(r'BREAKING GOALS|UEFA|NATIONS LEAGUE|PIALA DUNIA|WORLD CUP|HIGHLIGHTS|LIGA CHAMPIONS|PREMIER LEAGUE')), 'Olahraga', 'inews_sport')
put(has(r'INEWS|BULETIN|BREAKING NEWS|BERITA|HEADLINE|RAKYAT BERSUARA|SIKAT TIPIS|DEEP TALK|VIRAL TAPI HOAKS|\bVTH\b|INTERUPSI|BIKIN TERANG|IBREAK'), 'Berita', 'berita')
put(has(r'\bLIGA (?!BINTANG)|SUPER LEAGUE|PIALA|TIMNAS|FIFA|ASEAN (?:CUP|CHAMPIONSHIP)|PERSIB|PER5IB|PERSIJA|AREMA|PERSEBAYA|MOTOGP|BADMINTON|HASIL PERTANDINGAN|KICK OFF'), 'Olahraga', 'olahraga')
put(has(r'\bVS\b') & ~ch.isin(['iNews']), 'Olahraga', 'olahraga_vs')
put(has(r'CAHAYA HATI|KAJIAN|TAUSIYAH|GEMA RAMADAN|ISLAM ITU INDAH|AUDISI RELIGI|PENGAJIAN'), 'Religi', 'religi')
put(has(r"D['’]? ?ACADEMY|\bDA ?\d|IDA 2026|BAND ACADEMY|THE ICON|SPEKTAKULER|KONSER|AKSI INDOSIAR|AKSI 2026|BEFORE ON AIR|\bIDOL\b|AUDITION|KILAU RAYA|LIGA BINTANG JUARA|TOSI|DMD|PANGGUNG REZEKI|MASTERCHEF|PANTANG NGEMIS"), TALENT, 'talent')
put(has(r'GO ?SPOT|HOT ?SHOT|SILET|HALO SELEBRITI|INFOTAINMENT|BESTKISS|DAHSYATNYA'), 'Infotainment', 'infotainment')
put(has(r'UANG KAGET|BEDAH RUMAH|NEW FAMILY 100|DEG ?DEG-?AN|RUMAH TEKA TEKI|CAKEP CAKEP BAPER|1001 CARA MENGGAET|CEWEK CEWEK BADUNG|TAKE ME OUT|PAGI ?PAGI ?AMBYAR|BROWNIS|RUMPI|MONITOR KETUA|MAKAN RECEH|\bFYP\b|LAPOR PAK|PAS BUKA|JEJAK PETUALANG|TIBA[- ]TIBA KUIS|NGOBROL BARENG IDOLA|MANCUR|BIG IS BEAUTIFUL|COBOY JUNIOR|KARNAVAL|AWARDS|ANUGERAH LSF|SUPER DEAL|4 SEHAT 5 SULE|AWAS ADA SULE|DREAM ?BOX|ENAH BIKIN ENAK|ARISAN \(|UJUNG-UJUNGNYA DAMKAR|CUAN BOSS|BIKIN SENENG|FLOWER ISLAND|LEBIH SYANTIK|ISENGIN|NGEGHOSTING'), 'Variety & Lifestyle', 'variety')
put(has(r'\bFTV\b|KUASA IL+AHI|KIAN SANTANG|FATIH DI KAMPUNG JAWARA|CAHAYA HIKMAH|CAHAYA MENTARI|SULTAN AJI|TUTUR TINULAR|SUPARMAN REBORN|NYI RORO KIDUL|DZOLIM|MISTERI GUNUNG MERAPI|KISAH MISTERI|IPA ?& ?IPS|OBSESI|ARJUNA|RAJA DAN AKU|KINARA|DIBALIK RAHASIA|DUNIA KIARA|RAHASIA BATIN|MENCINTAI IPAR|TERBELENGGU RINDU|TERIKAT JANJI|KAU DITAKD[IR]+KAN UNTUKKU|DUSTA DIBALIK CINTA|\bAIRA\b|99 NAMA CINTA|CINTA SEPENUH JIWA|PUTRI UNTUK PANGERAN|TURUN RANJANG|\bKDU\b|MERANGKAI KISAH INDAH|KISAH NYATA SPESIAL|MUTIARA HATI|MENGETUK PINTU HATI|ASMARA GEN ?Z|BERI CINTA WAKTU|CINTA SELUAS SAMUDRA|PUTRI PEWARIS|ISTRI SATU-?SATUNYA|PESUGIHAN|SILUMAN|TUKANG OJEK PENGKOLAN|7 MANUSIA HARIMAU|PRASASTI|DEWI BINTARI|HATI HATI JATUH CINTA|DIA YANG TAK TERLIHAT|KISAH RUQYAHKU|KISAH LARA|SAMPEYAN MUSLIM'), 'Sinetron & FTV', 'sinetron')
# hashtag-only Shorts
put(has(r'ASMARAGENZ|BERICINTAWAKTU|BIARKANHATIBICARA|BUARKANHATIBICARA|PUTRIPEWARIS|ISTRISATUSATUNYA|JEJAKDUKADIANDRA|MENCINTAIIPAR|FTVMISTERI|SUPARMANREBORN|FATIHDIKAMPUNGJAWARA|CINTASEPENUHJIWA|TERBELENGGURINDU|TERIKATJANJI|KAUDITAKDIRKAN|MERANGKAIKISAHINDAH|KUASAILAHI|KIANSANTANG|7MANUSIAHARIMAU|DUSTADIBALIKCINTA|MUTIARAHATI', h), 'Sinetron & FTV', 'hashtag_sinetron')
put(has(r'THEICONINDONESIA|DACADEMY|DA8|BANDACADEMY|MASTERCHEF|TOSI|IDOLBAE', h), TALENT, 'hashtag_talent')
put(has(r'AGZLOVEFEST|PAGIPAGIAMBYAR|BROWNIS|RUMPI|MONITORKETUA|NOSECRET', h), 'Variety & Lifestyle', 'hashtag_variety')
put(has(r'DAHSYATNYAWEEKEND|GOSPOT|HOTSHOT', h), 'Infotainment', 'hashtag_info')
# generic episode markers (not for Trans channels)
put(has(r'\bEP(?:S|ISODE)?\.? ?#?\d|#EPS ?\d|\bPART\b|\(\d/\d\)') & ~ch.isin(TRANS), 'Sinetron & FTV', 'episode_marker')
# my guesses, please correct them (see the questions below)
UNSURE = r'KAMPUNG JAKARTA|KONON KATANYA|AYU ANAK DEPOK|KISAHKU|SENANDUNG|DUNIA TANPA BATAS|JOMBLO|SUPER DEDE|JANGAN CIUM GUE'
put(has(UNSURE), 'Sinetron & FTV', 'unsure_guess')
# channel defaults
put((ch == 'iNews') & (v['category_id'] == 25), 'Berita', 'default_inews')
put(ch.isin(TRANS) & has(r'\(\d{1,2}/\d{1,2}/\d{2}\)'), 'Variety & Lifestyle', 'default_trans_date')

In [ ]:
print(v.groupby('genre_rule').size().sort_values(ascending=False))
print(h[v['content_genre'] == 'Lainnya'].str.split().explode().value_counts().head(40))

genre_rule
8:Sinetron & FTV         26666
                         26464
7:Variety & Lifestyle    19108
1:Berita                 13206
9:Sinetron & FTV          6743
default_inews             6534
5:Talent Show & Musik     3031
2:Olahraga                2036
3:Olahraga                 896
0:Promo & Korporat         788
4:Religi                   435
6:Infotainment             396
dtype: int64
title
SHORTS                         8057
FTVMISTERI                     1948
ASMARAGENZ                      839
NOSECRET                        746
ASMARAGENZSCTV                  653
BERICINTAWAKTU                  409
TRANSTV                         273
THEICONINDONESIASCTV            250
MONITORKETUA                    230
WAJAHCINTAYANGLAINSCTV          186
MASTERCHEFINDONESIASEASON13     151
FATIHDIKAMPUNGJAWARA            138
BERICINTAWAKTUSCTV              126
IPAIPS                          123
BIARKANHATIBICARASCTV           117
SUPARMANREBORN                  116
MASTERCHEFINDONESIA   

In [ ]:
put(has(r'WAJAHCINTAYANGLAIN|CINTASEDALAMRINDU|ISTIQOMAHCINTA|CINTASELUASSAMUDRA|IPAIPS|DIAYANGTAKTERLIHAT|JAGONYAFILM|GTVBIGMOVIES|KEMBALINYARADENKIANSANTANG', h), 'Sinetron & FTV', 'hashtag_sinetron2')
put(has(r'HAFIZINDONESIA|LIGABINTANGJUARA|IA2026', h), TALENT, 'hashtag_talent2')
put(has(r'CAKEPCAKEPBAPER', h), 'Variety & Lifestyle', 'hashtag_variety2')

In [ ]:
import pandas as pd, numpy as np, re, unicodedata, warnings
warnings.filterwarnings('ignore', message='This pattern')

BASE   = '/content/drive/MyDrive/mnc-analytics/'
IN     = BASE + 'clean/yt_videos_clean_tahap1.csv'
OUT    = BASE + 'clean/yt_videos_clean_tahap2.csv'
SAMPLE = BASE + 'final/genre_validation_sample.csv'
v = pd.read_csv(IN)

def clean(s):
    s = unicodedata.normalize('NFKC', s).upper()
    s = re.sub(r'[\u200b-\u200f\u2060-\u2064\ufeff]', '', s)
    return re.sub(r'\s+', ' ', s).strip()

t  = v['title'].fillna('').map(clean)
h  = t.str.findall(r'#(\w+)').str.join(' ')
ch = v['channel_name']
TRANS  = ['Trans7', 'Trans TV']
TALENT = 'Talent & Reality Show'

genre = pd.Series('Lainnya', index=v.index)
rule  = pd.Series('', index=v.index)
done  = pd.Series(False, index=v.index)

def put(mask, g, label):
    mask = mask & ~done
    genre[mask], rule[mask], done[mask] = g, label, True

def has(words, src=None):
    pat = '|'.join(words)
    return (t if src is None else src).str.contains(pat, regex=True)

# ---------- keyword lists (short lines on purpose) ----------
PROMO = [
    'ALLO PAYLATER', 'TRANS (?:LUXURY )?HOTEL', 'JOGJA FINANCIAL',
    'JOGJA RUN', 'PADEL', 'GRAND RECRUITMENT', 'TRANS7 MENGAJAR',
    'SELAMAT BERBUKA', 'SAATNYA BERBUKA', 'UPACARA PENURUNAN BENDERA',
]
INEWS_SPORT = [
    'BREAKING GOALS', 'UEFA', 'NATIONS LEAGUE', 'PIALA DUNIA',
    'WORLD CUP', 'HIGHLIGHTS', 'LIGA CHAMPIONS', 'PREMIER LEAGUE',
]
BERITA = [
    'INEWS', 'BULETIN', 'BREAKING NEWS', 'BERITA', 'HEADLINE',
    'RAKYAT BERSUARA', 'SIKAT TIPIS', 'DEEP TALK', 'VIRAL TAPI HOAKS',
    r'\bVTH\b', 'INTERUPSI', 'BIKIN TERANG', 'IBREAK',
]
OLAHRAGA = [
    r'\bLIGA (?!BINTANG)', 'SUPER LEAGUE', 'PIALA', 'TIMNAS', 'FIFA',
    'ASEAN (?:CUP|CHAMPIONSHIP)', 'PERSIB', 'PER5IB', 'PERSIJA', 'AREMA',
    'PERSEBAYA', 'MOTOGP', 'BADMINTON', 'HASIL PERTANDINGAN', 'KICK OFF',
]
RELIGI = [
    'CAHAYA HATI', 'KAJIAN', 'TAUSIYAH', 'GEMA RAMADAN',
    'ISLAM ITU INDAH', 'AUDISI RELIGI', 'PENGAJIAN',
]
TALENT_KW = [
    "D['’]? ?ACADEMY", r'\bDA ?\d', 'IDA 2026', 'BAND ACADEMY', 'THE ICON',
    'SPEKTAKULER', 'KONSER', 'AKSI INDOSIAR', 'AKSI 2026', 'BEFORE ON AIR',
    r'\bIDOL\b', 'AUDITION', 'KILAU RAYA', 'LIGA BINTANG JUARA', 'TOSI',
    'DMD', 'PANGGUNG REZEKI', 'MASTERCHEF', 'PANTANG NGEMIS',
]
INFO = [
    'GO ?SPOT', 'HOT ?SHOT', 'SILET', 'HALO SELEBRITI',
    'INFOTAINMENT', 'BESTKISS', 'DAHSYATNYA',
]
VARIETY = [
    'UANG KAGET', 'BEDAH RUMAH', 'NEW FAMILY 100', 'DEG ?DEG-?AN',
    'RUMAH TEKA TEKI', 'CAKEP CAKEP BAPER', '1001 CARA MENGGAET',
    'CEWEK CEWEK BADUNG', 'TAKE ME OUT', 'PAGI ?PAGI ?AMBYAR', 'BROWNIS',
    'RUMPI', 'MONITOR KETUA', 'MAKAN RECEH', r'\bFYP\b', 'LAPOR PAK',
    'PAS BUKA', 'JEJAK PETUALANG', 'TIBA[- ]TIBA KUIS',
    'NGOBROL BARENG IDOLA', 'MANCUR', 'BIG IS BEAUTIFUL', 'COBOY JUNIOR',
    'KARNAVAL', 'AWARDS', 'ANUGERAH LSF', 'SUPER DEAL', '4 SEHAT 5 SULE',
    'AWAS ADA SULE', 'DREAM ?BOX', 'ENAH BIKIN ENAK', r'ARISAN \(',
    'UJUNG-UJUNGNYA DAMKAR', 'CUAN BOSS', 'BIKIN SENENG', 'FLOWER ISLAND',
    'LEBIH SYANTIK', 'ISENGIN', 'NGEGHOSTING', 'HUT ?SCTV',
]
SINETRON_A = [
    r'\bFTV\b', 'KUASA IL+AHI', 'KIAN SANTANG', 'FATIH DI KAMPUNG JAWARA',
    'CAHAYA HIKMAH', 'CAHAYA MENTARI', 'SULTAN AJI', 'TUTUR TINULAR',
    'SUPARMAN REBORN', 'NYI RORO KIDUL', 'DZOLIM', 'MISTERI GUNUNG MERAPI',
    'KISAH MISTERI', 'IPA ?& ?IPS', 'OBSESI', 'ARJUNA', 'RAJA DAN AKU',
    'KINARA', 'DIBALIK RAHASIA', 'DUNIA KIARA', 'RAHASIA BATIN',
]
SINETRON_B = [
    'MENCINTAI IPAR', 'TERBELENGGU RINDU', 'TERIKAT JANJI',
    'KAU DITAKD[IR]+KAN UNTUKKU', 'DUSTA DIBALIK CINTA', r'\bAIRA\b',
    '99 NAMA CINTA', 'CINTA SEPENUH JIWA', 'PUTRI UNTUK PANGERAN',
    'TURUN RANJANG', r'\bKDU\b', 'MERANGKAI KISAH INDAH',
    'KISAH NYATA SPESIAL', 'MUTIARA HATI', 'MENGETUK PINTU HATI',
    'ASMARA GEN ?Z', 'BERI CINTA WAKTU', 'CINTA SELUAS SAMUDRA',
]
SINETRON_C = [
    'PUTRI PEWARIS', 'ISTRI SATU-?SATUNYA', 'PESUGIHAN', 'SILUMAN',
    'TUKANG OJEK PENGKOLAN', '7 MANUSIA HARIMAU', 'PRASASTI',
    'DEWI BINTARI', 'HATI HATI JATUH CINTA', 'DIA YANG TAK TERLIHAT',
    'KISAH RUQYAHKU', 'KISAH LARA', 'SAMPEYAN MUSLIM',
]
# ---------- hashtag keyword lists ----------
H_SINETRON = [
    'ASMARAGENZ', 'BERICINTAWAKTU', 'BIARKANHATIBICARA',
    'BUARKANHATIBICARA', 'PUTRIPEWARIS', 'ISTRISATUSATUNYA',
    'JEJAKDUKADIANDRA', 'MENCINTAIIPAR', 'FTVMISTERI', 'SUPARMANREBORN',
    'FATIHDIKAMPUNGJAWARA', 'CINTASEPENUHJIWA', 'TERBELENGGURINDU',
    'TERIKATJANJI', 'KAUDITAKDIRKAN', 'MERANGKAIKISAHINDAH',
    'KUASAILAHI', 'KIANSANTANG', '7MANUSIAHARIMAU', 'DUSTADIBALIKCINTA',
    'MUTIARAHATI',
]
H_SINETRON2 = [
    'WAJAHCINTAYANGLAIN', 'CINTASEDALAMRINDU', 'ISTIQOMAHCINTA',
    'CINTASELUASSAMUDRA', 'IPAIPS', 'DIAYANGTAKTERLIHAT', 'JAGONYAFILM',
    'GTVBIGMOVIES', 'KEMBALINYARADENKIANSANTANG',
]
H_TALENT  = ['THEICONINDONESIA', 'DACADEMY', 'DA8', 'BANDACADEMY',
             'MASTERCHEF', 'TOSI', 'IDOLBAE']
H_TALENT2 = ['HAFIZINDONESIA', 'LIGABINTANGJUARA', 'IA2026']
H_VARIETY = ['AGZLOVEFEST', 'PAGIPAGIAMBYAR', 'BROWNIS', 'RUMPI',
             'MONITORKETUA', 'NOSECRET']
H_VARIETY2 = ['CAKEPCAKEPBAPER', 'HUTSCTV']
H_INFO    = ['DAHSYATNYAWEEKEND', 'GOSPOT', 'HOTSHOT']

# ---------- apply rules (first match wins) ----------
put(has(PROMO), 'Promo & Korporat', 'promo')
put((ch == 'iNews') & ((v['category_id'] == 17) | has(INEWS_SPORT)),
    'Olahraga', 'inews_sport')
put(has(BERITA), 'Berita', 'berita')
put(has(OLAHRAGA), 'Olahraga', 'olahraga')
put(has([r'\bVS\b']) & ~ch.isin(['iNews']), 'Olahraga', 'olahraga_vs')
put(has(RELIGI), 'Religi', 'religi')
put(has(TALENT_KW), TALENT, 'talent')
put(has(INFO), 'Infotainment', 'infotainment')
put(has(VARIETY), 'Variety & Lifestyle', 'variety')
put(has(SINETRON_A + SINETRON_B + SINETRON_C), 'Sinetron & FTV', 'sinetron')

put(has(H_SINETRON, h), 'Sinetron & FTV', 'hashtag_sinetron')
put(has(H_TALENT, h), TALENT, 'hashtag_talent')
put(has(H_VARIETY, h), 'Variety & Lifestyle', 'hashtag_variety')
put(has(H_INFO, h), 'Infotainment', 'hashtag_info')
put(has(H_SINETRON2, h), 'Sinetron & FTV', 'hashtag_sinetron2')
put(has(H_TALENT2, h), TALENT, 'hashtag_talent2')
put(has(H_VARIETY2, h), 'Variety & Lifestyle', 'hashtag_variety2')

EPISODE = [r'\bEP(?:S|ISODE)?\.? ?#?\d', r'#EPS ?\d', r'\bPART\b', r'\(\d/\d\)']
put(has(EPISODE) & ~ch.isin(TRANS), 'Sinetron & FTV', 'episode_marker')

UNSURE = ['KAMPUNG JAKARTA', 'KONON KATANYA', 'AYU ANAK DEPOK', 'KISAHKU',
          'SENANDUNG', 'DUNIA TANPA BATAS', 'JOMBLO', 'SUPER DEDE',
          'JANGAN CIUM GUE']
put(has(UNSURE), 'Sinetron & FTV', 'unsure_guess')
put((ch == 'iNews') & (v['category_id'] == 25), 'Berita', 'default_inews')
put(ch.isin(TRANS) & has([r'\(\d{1,2}/\d{1,2}/\d{2}\)']),
    'Variety & Lifestyle', 'default_trans_date')

# generic-tag-only videos can't be identified
GENERIC = {'SHORTS', 'SHORT', 'TRANSTV', 'INDOSIAR', 'GTV', 'SCTV',
           'RCTI', 'MNCTV', 'INEWS'}
only_generic = h.ne('') & h.str.split().map(lambda x: set(x) <= GENERIC)
m = (genre == 'Lainnya') & only_generic
genre[m], rule[m] = 'Tidak teridentifikasi', 'generic_tag_only'

v['content_genre'], v['genre_rule'] = genre, rule
v['is_promo'] = v['content_genre'].eq('Promo & Korporat')

# ---------- checks ----------
print('shape:', v.shape)
print('\ngenre_rule counts:')
print(v['genre_rule'].replace('', 'NO_MATCH').value_counts().to_string())
print('\ngenre share per channel (%):')
print((pd.crosstab(v['channel_name'], v['content_genre'],
                   normalize='index') * 100).round(1).to_string())
ns = v[v['format'] != 'Shorts']
print('\nLainnya share, non-Shorts only (%):')
print((ns['content_genre'].eq('Lainnya')
       .groupby(ns['channel_name']).mean() * 100).round(1))
print('\ngenre x format:')
print(pd.crosstab(v['content_genre'], v['format']))

un = v[v['content_genre'] == 'Lainnya']
print('\ntop remaining hashtags:')
print(h[un.index].str.split().explode().value_counts().head(40))
segs = un['title'].str.upper().str.split('|').explode().str.strip()
print('\ntop unmatched segments:')
print(segs[~segs.str.contains(r'\d{1,2}[/ ]\d|#SHORTS', regex=True,
                              na=False)].value_counts().head(30))
for c, g in un.groupby('channel_name'):
    print(f'\n--- unmatched sample: {c} ---')
    print(g['title'].sample(min(6, len(g)), random_state=3).tolist())

# 100-row validation sample (identified videos only)
s = v[v['content_genre'] != 'Tidak teridentifikasi'].sample(
    100, random_state=42)[['video_id', 'channel_name', 'title',
                           'content_genre', 'genre_rule']].copy()
s['genre_manual'] = ''
s.to_csv(SAMPLE, index=False)
v.to_csv(OUT, index=False)

shape: (106303, 37)

genre_rule counts:
genre_rule
sinetron              28807
variety               22149
berita                13148
default_inews          6487
NO_MATCH               5838
episode_marker         5240
hashtag_sinetron       5030
default_trans_date     4741
talent                 4383
olahraga               1950
generic_tag_only       1791
hashtag_variety         988
religi                  968
unsure_guess            897
olahraga_vs             896
promo                   806
hashtag_sinetron2       799
infotainment            682
hashtag_talent          255
inews_sport             238
hashtag_talent2         172
hashtag_variety2         38

genre share per channel (%):
content_genre  Berita  Infotainment  Lainnya  Olahraga  Promo & Korporat  Religi  Sinetron & FTV  Talent & Reality Show  Tidak teridentifikasi  Variety & Lifestyle
channel_name                                                                                                                               

In [ ]:
# ---------- final patch ----------
PROMO2 = ['RAMADAN SUPER SALE', 'LIVE SHOPPING', 'MERCHANT CT CORP',
          'SAKSIKAN SPONGEBOB']
RELIGI2 = ['CERAMAH', 'MAMAH DEDEH', 'CAHAYA TAUHID', 'KISAH PARA RASUL',
           'CAHAYA MUDA INDONESIA', r'\bCHI\b', 'KISAH WALI SONGO',
           'PARA PENCARI TUHAN', 'SIDANG ISBAT', 'PINTU BERKAH']
TALENT2 = [r'IDA 20\d\d', r'\bAKSI\b', r'\bKDI\b', 'HAFIZ INDONESIA',
           'SALUTE TO', 'GEMPITA', 'PKD', 'RTKR', 'SUARA DEMI IMPIAN']
VARIETY2 = ['MANDA CURHAT', 'CAGUR', 'DAPUR NGEBOR', 'TUKANG MAKAN',
            'HUT AMAZING', 'HAPPY NEW YEAR', 'PESTA RAYA', 'PROKLAMASI',
            'UNTOLD STORY', 'BANG UDIN', 'SEA GAMES']
INFO2 = ['THE WEDDING OF', 'THE ENGAGEMENT OF', 'LESLAR', 'GENDER REVEAL',
         'EXTENDED BACKSTAGE', 'PEMAKAMAN']
SINETRON2 = ['PENGAKUAN TERLARANG', 'SUARA HATI ISTRI', 'LIVE MISTERI',
             'PRAHARA', 'SERIGALA BUCIN', 'BUKAN CINDERELLA',
             'MURNI I LAP YU', 'NIKAH DULUAN', 'TIBA TIBA SETAN']
H_SIN3 = ['MISTERIGUNUNGMERAPI', 'FTVSCTV', 'LAUTANCINTA', 'SEBENINGCINTA',
          'CINTADIBAWAHTANGAN', 'CERITACINTAKITA', 'LORONGWAKTU',
          'SAMBALADO', 'CAHAYAHIKMAH', 'PETUALANGANRAHASIAQUEENA',
          'VISIONPLUSORIGINALS', 'TIBATIBASETAN']
H_VAR3 = ['ARISAN', 'LAPORPAK', 'SPILLDDTEA', 'HAHAHIHI', 'MANDACURHAT',
          'AGZTHECONCERT', 'ALLSTARSSPORTFEST', 'AMAZING23', 'HUTMNCTV',
          'NEWFAMILY100']
H_TAL3 = ['KILAURAYA', 'ROADTOKILAURAYA', 'ATA20']
H_REL3 = ['PARAPENCARITUHAN', 'MAMAHDEDEH', 'CAHAYATAUHID', 'GEMARAMADAN']
H_INF3 = ['DAHSYAT', 'AMANDAMANOPO']

put(has(PROMO2), 'Promo & Korporat', 'promo2')
put(has(RELIGI2), 'Religi', 'religi2')
put(has(TALENT2), TALENT, 'talent2')
put(has(VARIETY2), 'Variety & Lifestyle', 'variety2')
put(has(INFO2), 'Infotainment', 'infotainment2')
put(has(SINETRON2), 'Sinetron & FTV', 'sinetron2')
put(has(H_SIN3, h), 'Sinetron & FTV', 'hashtag_sinetron3')
put(has(H_VAR3, h), 'Variety & Lifestyle', 'hashtag_variety3')
put(has(H_TAL3, h), TALENT, 'hashtag_talent3')
put(has(H_REL3, h), 'Religi', 'hashtag_religi3')
put(has(H_INF3, h), 'Infotainment', 'hashtag_info3')

In [ ]:
# reapply generic-tag rule to anything still unmatched
GENERIC = {'SHORTS', 'SHORT', 'TRANSTV', 'INDOSIAR', 'GTV', 'SCTV',
           'RCTI', 'MNCTV', 'INEWS'}
only_generic = h.ne('') & h.str.split().map(lambda x: set(x) <= GENERIC)
m = (genre == 'Lainnya') & only_generic
genre[m], rule[m] = 'Tidak teridentifikasi', 'generic_tag_only'

v['content_genre'], v['genre_rule'] = genre, rule
v['is_promo'] = v['content_genre'].eq('Promo & Korporat')

# checks
print('rows:', len(v))
print('\ncounts per genre:')
print(v['content_genre'].value_counts().to_string())
ns = v[v['format'] != 'Shorts']
print('\nLainnya share, non-Shorts only (%):')
print((ns['content_genre'].eq('Lainnya')
       .groupby(ns['channel_name']).mean() * 100).round(1))
print('\nTidak teridentifikasi per channel:')
print(v[v['content_genre'] == 'Tidak teridentifikasi']
      .groupby('channel_name').size())

# save + regenerate the 100-row validation sample
s = v[v['content_genre'] != 'Tidak teridentifikasi'].sample(
    100, random_state=42)[['video_id', 'channel_name', 'title',
                           'content_genre', 'genre_rule']].copy()
s['genre_manual'] = ''
s.to_csv(SAMPLE, index=False)
v.to_csv(OUT, index=False)
print('\nsaved:', OUT)

rows: 106303

counts per genre:
content_genre
Sinetron & FTV           41097
Variety & Lifestyle      28661
Berita                   19635
Talent & Reality Show     5112
Lainnya                   3695
Olahraga                  3084
Tidak teridentifikasi     1783
Religi                    1616
Promo & Korporat           839
Infotainment               781

Lainnya share, non-Shorts only (%):
channel_name
GTV         0.6
Indosiar    5.0
MNCTV       1.2
RCTI        7.4
SCTV        2.0
Trans TV    2.9
Trans7      1.2
iNews       0.0
Name: content_genre, dtype: float64

Tidak teridentifikasi per channel:
channel_name
GTV            1
Indosiar    1757
MNCTV          1
RCTI           1
SCTV          22
Trans7         1
dtype: int64

saved: /content/drive/MyDrive/mnc-analytics/clean/yt_videos_clean_tahap2.csv


In [ ]:
import pandas as pd
p = '/content/drive/MyDrive/mnc-analytics/final/genre_validation_sample.csv'

try:
    s = pd.read_csv(p, sep=';', encoding='utf-8-sig')
except UnicodeDecodeError:
    s = pd.read_csv(p, sep=';', encoding='cp1252')

print(s.shape)                 # expect (100, 6)
print(s.columns.tolist())      # expect video_id ... genre_manual

(100, 6)
['video_id', 'channel_name', 'title', 'content_genre', 'genre_rule', 'genre_manual']


In [ ]:
s['genre_manual'] = s['genre_manual'].astype(str).str.strip().replace('nan', pd.NA)

print('labeled:', s['genre_manual'].notna().sum(), '/ 100')
print('\nyour label names:')
print(s['genre_manual'].value_counts(dropna=False).to_string())

mapping = {
    # 'your name': 'my label',
}
s['genre_manual'] = s['genre_manual'].replace(mapping)
s['correct'] = s['content_genre'] == s['genre_manual']

print(f"\noverall accuracy: {s['correct'].mean():.0%}")
print('\naccuracy per rule:')
print(s.groupby('genre_rule')['correct'].agg(['mean', 'size'])
      .sort_values('size', ascending=False).to_string())
print('\nconfusion (rows = auto label, cols = your label):')
print(pd.crosstab(s['content_genre'], s['genre_manual']).to_string())
print('\nmislabeled rows:')
print(s.loc[~s['correct'], ['channel_name', 'title', 'content_genre',
                            'genre_manual']].to_string())

labeled: 100 / 100

your label names:
genre_manual
Sinetron & FTV           39
Variety & Lifestyle      24
Berita                   18
Talent & Reality Show     9
Infotainment              5
Olahraga                  2
Religi                    2
Lainnya                   1

overall accuracy: 81%

accuracy per rule:
                        mean  size
genre_rule                        
sinetron            0.966667    30
variety             0.700000    20
berita              1.000000    13
episode_marker      0.833333     6
talent              0.666667     6
default_inews       1.000000     5
default_trans_date  0.800000     5
infotainment        0.333333     3
olahraga_vs         0.500000     2
infotainment2       0.000000     1
hashtag_sinetron    1.000000     1
hashtag_variety     0.000000     1
religi              1.000000     1
olahraga            1.000000     1
sinetron2           1.000000     1
talent2             1.000000     1

confusion (rows = auto label, cols = your label):
g

In [ ]:
def override(mask, g, label):
    genre[mask], rule[mask] = g, label

SIN_ALL = SINETRON_A + SINETRON_B + SINETRON_C

override(has(['BEDAH RUMAH', 'UANG KAGET', 'TAKE ME OUT']), TALENT, 'fix_reality')
override(has([r'\| ?OBSESI\b', r'\bFYP \(', r'\bOTW \(']) | has(['NOSECRET'], h),
         'Infotainment', 'fix_info')
override(has(['DAHSYATNYA', 'EXTENDED BACKSTAGE', 'PESTA RAYA']) | has(['DAHSYATNYAWEEKEND'], h),
         'Variety & Lifestyle', 'fix_variety')
override(has(['HAFI[DZ]+ INDONESIA']) | has(['HAFI[DZ]+INDONESIA'], h), 'Religi', 'fix_religi')
override(has(['BIG IS BEAUTIFUL']), 'Sinetron & FTV', 'fix_sinetron')
override((rule == 'olahraga_vs') & has(SIN_ALL), 'Sinetron & FTV', 'fix_vs_sinetron')

v['content_genre'], v['genre_rule'] = genre, rule
v['is_promo'] = v['content_genre'].eq('Promo & Korporat')
print(v['content_genre'].value_counts().to_string())
print('\nfix rows:', v['genre_rule'].str.startswith('fix_').sum())
v.to_csv(OUT, index=False)

# accuracy on the SAME sample after fixes (optimistic, for info only)
s2 = s.drop(columns='content_genre').merge(v[['video_id', 'content_genre']], on='video_id')
print(f"\nsame-sample accuracy after fixes: {(s2['content_genre'] == s2['genre_manual']).mean():.0%}")

content_genre
Sinetron & FTV           40904
Variety & Lifestyle      24254
Berita                   19624
Talent & Reality Show     7652
Lainnya                   3694
Infotainment              2948
Olahraga                  2765
Religi                    1846
Tidak teridentifikasi     1783
Promo & Korporat           833

fix rows: 6084

same-sample accuracy after fixes: 96%


In [ ]:
override(has([r'\bOBSESI\b']), 'Infotainment', 'fix_info')
v['content_genre'], v['genre_rule'] = genre, rule
print((v['genre_rule'] == 'fix_info').sum())
v.to_csv(OUT, index=False)

2665


In [ ]:
import pandas as pd, re
from collections import Counter

r = pd.read_csv('/content/drive/MyDrive/mnc-analytics/clean/app_reviews_clean_tahap1.csv')

SLANG = {'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak',
         'tdk': 'tidak', 'g': 'tidak', 'bgt': 'banget', 'yg': 'yang', 'dgn': 'dengan',
         'udh': 'sudah', 'udah': 'sudah', 'sdh': 'sudah', 'blm': 'belum', 'tp': 'tapi',
         'krn': 'karena', 'jd': 'jadi', 'aja': 'saja', 'sy': 'saya', 'gw': 'saya',
         'gue': 'saya', 'aplikasi': 'aplikasi', 'apk': 'aplikasi', 'app': 'aplikasi',
         'iklannya': 'iklan', 'ngelag': 'lag', 'ngeleg': 'lag', 'lemot': 'lambat',
         'ngebuffer': 'buffering', 'buffer': 'buffering'}
STOP = set('yang dan di ke dari ini itu saya aplikasi tidak bisa ada untuk sudah '
           'aja saja juga mau kalau kok lagi apa tapi jadi sama dengan banget '
           'nya karena masih tolong sangat lebih harus kami kenapa pas buat'.split())

def norm(x):
    x = str(x).lower()
    x = re.sub(r'http\S+|www\.\S+', ' ', x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)           # drops emoji & punctuation
    return ' '.join(SLANG.get(w, w) for w in x.split())

r['content_clean'] = r['content_raw'].map(norm)
neg = r[r['score'] <= 2]

for app, g in neg.groupby('app'):
    words, bigrams = Counter(), Counter()
    for txt in g['content_clean']:
        w = txt.split()
        words.update(x for x in w if x not in STOP and len(x) > 2)
        bigrams.update(f'{a} {b}' for a, b in zip(w, w[1:])
                       if a not in STOP or b not in STOP)
    print(f'\n===== {app}  (negative reviews: {len(g)}) =====')
    print('words  :', [f'{k}({v})' for k, v in words.most_common(40)])
    print('bigrams:', [f'{k}({v})' for k, v in bigrams.most_common(30)])

print('\nsample negative reviews:')
print(neg.sample(15, random_state=7)[['app', 'content_clean']].to_string())


===== RCTIplus  (negative reviews: 7857) =====
words  : ['nonton(2443)', 'bayar(1645)', 'iklan(1144)', 'malah(881)', 'rcti(646)', 'beli(635)', 'timnas(621)', 'padahal(585)', 'bola(568)', 'paket(531)', 'live(526)', 'banyak(456)', 'jelek(445)', 'langganan(440)', 'lag(432)', 'jelas(403)', 'terus(376)', 'download(373)', 'berbayar(355)', 'berlangganan(352)', 'bagus(333)', 'premium(315)', 'streaming(308)', 'parah(302)', 'buruk(283)', 'suruh(277)', 'eror(271)', 'siaran(259)', 'indonesia(257)', 'gabisa(256)', 'update(254)', 'error(252)', 'dulu(252)', 'sering(242)', 'mulu(236)', 'kalo(236)', 'lambat(235)', 'baru(233)', 'susah(229)', 'buka(224)']
bigrams: ['sudah bayar(638)', 'mau nonton(594)', 'bisa nonton(465)', 'tidak jelas(353)', 'beli paket(340)', 'nonton timnas(327)', 'nonton bola(303)', 'sudah beli(223)', 'harus bayar(220)', 'buat nonton(217)', 'malah tidak(186)', 'nonton live(173)', 'tidak guna(160)', 'banyak iklan(151)', 'sudah berlangganan(142)', 'sudah langganan(131)', 'suruh bayar(1

In [ ]:
import pandas as pd, re

BASE = '/content/drive/MyDrive/mnc-analytics/'
r = pd.read_csv(BASE + 'clean/app_reviews_clean_tahap1.csv')

SLANG = {
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak',
    'ngga': 'tidak', 'tdk': 'tidak', 'g': 'tidak', 'bgt': 'banget',
    'yg': 'yang', 'dgn': 'dengan', 'udh': 'sudah', 'udah': 'sudah',
    'sdh': 'sudah', 'blm': 'belum', 'tp': 'tapi', 'krn': 'karena',
    'jd': 'jadi', 'aja': 'saja', 'sy': 'saya', 'gw': 'saya',
    'gue': 'saya', 'apk': 'aplikasi', 'app': 'aplikasi',
    'gabisa': 'tidak bisa', 'gbisa': 'tidak bisa', 'gbs': 'tidak bisa',
    'gakbisa': 'tidak bisa', 'eror': 'error', 'erorr': 'error',
    'errorr': 'error', 'ngelag': 'lag', 'ngeleg': 'lag', 'lemot': 'lambat',
    'ngebuffer': 'buffering', 'buffer': 'buffering', 'jls': 'jelas',
    'iklannya': 'iklan', 'd': 'di',
}

def norm(x):
    x = str(x).lower()
    x = re.sub(r'http\S+|www\.\S+', ' ', x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    return ' '.join(SLANG.get(w, w) for w in x.split())

r['content_clean'] = r['content_raw'].map(norm)
c = r['content_clean']

def kw(words):
    return c.str.contains(r'\b(?:' + '|'.join(words) + r')\b', regex=True)

PAY_TERM = ['bayar', 'beli', 'langganan', 'berlangganan', 'premium',
            'paket', 'platinum', 'vip', 'subscribe']
FAIL = ['tidak bisa', 'belum aktif', 'tidak aktif', 'tidak muncul',
        'tetap tidak', 'malah', 'terkunci', 'disuruh', 'suruh',
        'beli lagi', 'bayar lagi', 'percuma']

TOPICS = {
 'pembayaran': kw(['saldo', 'pulsa', 'dana', 'gopay', 'ovo', 'shopeepay',
                   'linkaja', 'terpotong', 'dipotong', 'potong', 'refund',
                   'uang saya', 'duit saya', 'otomatis', 'perpanjang',
                   'diperpanjang', 'tagihan', 'transaksi', 'konfirmasi',
                   'gagal bayar', 'tidak bisa bayar', 'voucher',
                   'kartu kredit', 'metode pembayaran']),
 'akses_setelah_bayar': kw(PAY_TERM) & kw(FAIL),
 'login_akun': kw(['login', 'log in', 'logout', 'log out', 'masuk akun',
                   'tidak bisa masuk', 'akun', 'otp', 'verifikasi',
                   'password', 'kata sandi', 'daftar', 'registrasi']),
 'error_crash': kw(['error', 'crash', 'force close', 'keluar sendiri',
                    'tidak bisa dibuka', 'tidak bisa di buka', 'blank',
                    'layar hitam', 'layar putih', 'hang', 'bug',
                    'not responding']),
 'lag_buffering': kw(['lag', 'lambat', 'buffering', 'loading',
                      'putus putus', 'patah patah', 'macet', 'delay',
                      'freeze', 'ngefreeze', 'jaringan', 'koneksi',
                      'muter muter', 'stuck', 'tersendat']),
 'iklan': c.str.contains(r'\biklan\w*', regex=True),
 'konten': kw(['film', 'series', 'serial', 'episode', 'eps',
               'tidak lengkap', 'kurang lengkap', 'replay',
               'tayangan ulang', 'siaran ulang', 'kepotong', 'durasi',
               'channel', 'saluran', 'kualitas gambar', 'resolusi',
               'buram', 'burik', 'subtitle', 'konten']),
 'paywall_harga': kw(['harus bayar', 'suruh bayar', 'disuruh bayar',
                      'berbayar', 'mahal', 'kemahalan', 'dulu gratis',
                      'tidak gratis']),
 'tv_device': kw(['ke tv', 'smart tv', 'android tv', 'tv box', 'cast',
                  'chromecast', 'mirroring', 'stb', 'set top box']),
}
for k, m in TOPICS.items():
    r['is_' + k] = m

r['mentions_sport'] = kw(['timnas', 'bola', 'sepak bola', 'liga', 'piala',
                          'sport', 'sports', 'pertandingan', 'match',
                          'motogp', 'afc', 'aff', 'fifa', 'badminton',
                          'bulutangkis', 'ucl', 'premier league'])

flag_cols = ['is_' + k for k in TOPICS]
r['n_topics'] =

SyntaxError: invalid syntax (944916817.py, line 78)

In [ ]:
exec(open('/content/drive/MyDrive/mnc-analytics/complaint_topic.py').read())

shape: (46649, 35)

% of NEGATIVE reviews mentioning each topic (multi-label):
app                     RCTIplus  Vidio  Visionplus
is_pembayaran                1.7   13.2         6.4
is_akses_setelah_bayar      16.6   21.8        41.6
is_login_akun                4.8    5.6        12.3
is_error_crash               9.1    6.4        12.1
is_lag_buffering            12.6   12.0         7.1
is_iklan                    10.2   17.3         3.6
is_konten                    5.3   12.2         6.7
is_paywall_harga            10.6    8.2         4.6
is_tv_device                 0.4    0.7         3.3
mentions_sport              16.0    8.1        22.7

primary topic, negative reviews (%):
app                  RCTIplus  Vidio  Visionplus
complaint_topic                                 
akses_setelah_bayar      15.8   18.2        38.3
error_crash               6.6    4.2         4.9
iklan                     7.3   14.0         2.9
konten                    2.8    4.8         2.9
lag_buffering    

In [ ]:
import pandas as pd, re

BASE = '/content/drive/MyDrive/mnc-analytics/'
r = pd.read_csv(BASE + 'clean/app_reviews_clean_tahap1.csv')

SLANG = {
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak',
    'ngga': 'tidak', 'ngak': 'tidak', 'ngg': 'tidak', 'engga': 'tidak',
    'enggak': 'tidak', 'tdk': 'tidak', 'tdak': 'tidak', 'g': 'tidak',
    'bgt': 'banget', 'yg': 'yang', 'dgn': 'dengan', 'udh': 'sudah',
    'udah': 'sudah', 'sdh': 'sudah', 'blm': 'belum', 'tp': 'tapi',
    'krn': 'karena', 'jd': 'jadi', 'aja': 'saja', 'sy': 'saya',
    'gw': 'saya', 'gua': 'saya', 'gue': 'saya', 'apk': 'aplikasi',
    'app': 'aplikasi', 'gabisa': 'tidak bisa', 'gbisa': 'tidak bisa',
    'gbs': 'tidak bisa', 'gakbisa': 'tidak bisa',
    'gabole': 'tidak boleh', 'eror': 'error', 'erorr': 'error',
    'errorr': 'error', 'ngelag': 'lag', 'ngeleg': 'lag',
    'lemot': 'lambat', 'ngebuffer': 'buffering', 'buffer': 'buffering',
    'jls': 'jelas', 'iklannya': 'iklan', 'd': 'di', 'nnton': 'nonton',
    'nntn': 'nonton', 'ntn': 'nonton', 'chanel': 'channel',
    'chanell': 'channel', 'cenel': 'channel',
    'livestream': 'live streaming', 'livestreaming': 'live streaming',
    'ksih': 'kasih', 'gmn': 'gimana', 'knp': 'kenapa',
    'knpa': 'kenapa', 'instal': 'install', 'uninstal': 'uninstall',
    'apdet': 'update',
}

def norm(x):
    x = str(x).lower()
    x = re.sub(r'http\S+|www\.\S+', ' ', x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'([a-z])\1{2,}', r'\1', x)
    return ' '.join(SLANG.get(w, w) for w in x.split())

r['content_clean'] = r['content_raw'].map(norm)
c = r['content_clean']

def kw(words):
    pat = r'\b(?:' + '|'.join(words) + r')\b'
    return c.str.contains(pat, regex=True)

PAY_TERM = ['bayar', 'beli', 'langganan', 'berlangganan',
            'premium', 'paket', 'platinum', 'vip', 'subscribe']
FAIL = ['tidak bisa', 'belum aktif', 'tidak aktif',
        'tidak muncul', 'tetap tidak', 'malah', 'terkunci',
        'disuruh', 'suruh', 'beli lagi', 'bayar lagi', 'percuma']

PEMBAYARAN = [
    'saldo', 'pulsa', 'dana', 'gopay', 'ovo', 'shopeepay',
    'linkaja', 'terpotong', 'dipotong', 'potong', 'refund',
    'uang saya', 'duit saya', 'otomatis', 'perpanjang',
    'diperpanjang', 'tagihan', 'transaksi', 'konfirmasi',
    'gagal bayar', 'tidak bisa bayar', 'voucher', 'kartu kredit',
    'metode pembayaran', 'berhenti langganan',
    'berhenti berlangganan', 'batal', 'batalkan', 'membatalkan',
    'pembatalan', 'unsubscribe', 'cancel', '2 kali lipat',
]
LOGIN = [
    'login', 'log in', 'logout', 'log out', 'masuk akun',
    'tidak bisa masuk', 'akun', 'otp', 'verifikasi', 'password',
    'kata sandi', 'daftar', 'registrasi',
]
ERROR = [
    'error', 'crash', 'force close', 'keluar sendiri',
    'tidak bisa dibuka', 'tidak bisa di buka', 'blank',
    'layar hitam', 'layar putih', 'hang', 'bug', 'not responding',
    'update', 'diupdate', 'install', 'uninstall', 'penyimpanan',
    'memori',
]
GAGAL_PUTAR = [
    'tidak bisa nonton', 'tidak bisa menonton', 'tidak bisa ditonton',
    'tidak bisa di tonton', 'tidak bisa diputar',
    'tidak bisa di putar', 'tidak bisa muter', 'tidak bisa buat muter',
    'tidak bisa live', 'tidak bisa streaming', 'tidak ada gambar',
    'hilang gambar', 'gambar hilang', 'gambarnya hilang',
    'layar gelap', 'gangguan', 'rusak', 'tidak bisa menikmati',
    'tidak bisa diakses', 'tidak bisa akses', 'tidak keluar gambar',
    'tidak bisa play',
]
LAG = [
    'lag', 'lambat', 'buffering', 'loading', 'putus putus',
    'patah patah', 'macet', 'delay', 'freeze', 'ngefreeze',
    'jaringan', 'koneksi', 'muter muter', 'stuck', 'tersendat',
]
KONTEN = [
    'film', 'series', 'serial', 'episode', 'eps', 'tidak lengkap',
    'kurang lengkap', 'replay', 'tayangan ulang', 'siaran ulang',
    'kepotong', 'durasi', 'channel', 'saluran', 'kualitas gambar',
    'resolusi', 'buram', 'burik', 'subtitle', 'konten',
    'tidak menyiarkan', 'tidak disiarkan', 'tidak tayang',
]
PAYWALL = [
    'harus bayar', 'suruh bayar', 'disuruh bayar', 'berbayar',
    'mahal', 'kemahalan', 'dulu gratis', 'tidak gratis', 'gratis',
    'harus premium', 'harus langganan', 'harus berlangganan',
    'saja bayar', 'taunya bayar', 'kirain gratis',
]
TV = [
    'ke tv', 'smart tv', 'android tv', 'tv box', 'cast',
    'chromecast', 'mirroring', 'stb', 'set top box',
]
SPORT = [
    'timnas', 'bola', 'sepak bola', 'sepakbola', 'liga', 'piala',
    'sport', 'sports', 'pertandingan', 'match', 'motogp', 'afc',
    'aff', 'fifa', 'badminton', 'bulutangkis', 'ucl', 'persib',
    'persija', 'premier league',
]

TOPICS = {
    'pembayaran': kw(PEMBAYARAN),
    'akses_setelah_bayar': kw(PAY_TERM) & kw(FAIL),
    'login_akun': kw(LOGIN),
    'error_crash': kw(ERROR),
    'gagal_putar': kw(GAGAL_PUTAR),
    'lag_buffering': kw(LAG),
    'iklan': c.str.contains(r'\biklan\w*', regex=True),
    'konten': kw(KONTEN),
    'paywall_harga': kw(PAYWALL),
    'tv_device': kw(TV),
}
for k, m in TOPICS.items():
    r['is_' + k] = m
r['mentions_sport'] = kw(SPORT)

flag_cols = ['is_' + k for k in TOPICS]
r['n_topics'] = r[flag_cols].sum(axis=1)
r['n_words'] = c.str.split().str.len()

PRIORITY = list(TOPICS)            # pembayaran first, tv_device last
r['complaint_topic'] = 'lainnya'
for k in reversed(PRIORITY):
    r.loc[r['is_' + k], 'complaint_topic'] = k

print('done:', r.shape)

done: (46649, 37)


In [ ]:
neg = r[r['score'] <= 2]
long_neg = neg[neg['n_words'] >= 6]

print('% of NEGATIVE reviews per topic (multi-label):')
t1 = neg.groupby('app')[flag_cols + ['mentions_sport']].mean()
print((t1 * 100).round(1).T.to_string())

print('\nprimary topic, negative reviews (%):')
t2 = pd.crosstab(neg['complaint_topic'], neg['app'],
                 normalize='columns')
print((t2 * 100).round(1).to_string())

print('\nuntagged negative (%), all vs 6+ words:')
u_all = neg['complaint_topic'].eq('lainnya').groupby(neg['app']).mean()
u_long = (long_neg['complaint_topic'].eq('lainnya')
          .groupby(long_neg['app']).mean())
print(pd.DataFrame({'all': u_all, '6plus_words': u_long})
      .mul(100).round(1).to_string())

print('\nspike vs normal days, primary topic, negative (%):')
t3 = pd.crosstab(neg['complaint_topic'], neg['is_spike_day'],
                 normalize='columns')
print((t3 * 100).round(1).to_string())

print('\nsample untagged negative reviews (6+ words):')
un = long_neg[long_neg['complaint_topic'] == 'lainnya']
print(un.sample(25, random_state=5)[['app', 'content_clean']]
      .to_string())

vs = neg.sample(100, random_state=42)[
    ['review_id', 'app', 'score', 'content_clean',
     'complaint_topic']].copy()
vs['complaint_manual'] = ''
vs.to_csv(BASE + 'final/complaint_validation_sample.csv',
          index=False, sep=';', encoding='utf-8-sig')
r.to_csv(BASE + 'clean/app_reviews_clean.csv', index=False)
print('\nsaved')

% of NEGATIVE reviews per topic (multi-label):
app                     RCTIplus  Vidio  Visionplus
is_pembayaran                1.9   14.1         7.3
is_akses_setelah_bayar      17.1   23.0        43.7
is_login_akun                4.8    5.7        12.3
is_error_crash              14.1    9.4        17.0
is_gagal_putar              14.5   13.2        26.0
is_lag_buffering            14.9   13.5         8.1
is_iklan                    10.2   17.3         3.6
is_konten                    6.3   13.3         8.3
is_paywall_harga            13.5   11.6         7.0
is_tv_device                 5.1    6.6         7.5
mentions_sport              16.7    8.7        23.8

primary topic, negative reviews (%):
app                  RCTIplus  Vidio  Visionplus
complaint_topic                                 
akses_setelah_bayar      16.2   18.7        39.8
error_crash              10.7    6.2         7.1
gagal_putar               5.5    4.0         2.7
iklan                     6.6   13.4         2

In [ ]:
SLANG.update({
    'nonto': 'nonton', 'byar': 'bayar', 'bsa': 'bisa', 'bs': 'bisa',
    'nggk': 'tidak', 'ngk': 'tidak', 'kaga': 'tidak', 'kga': 'tidak',
    'mlhn': 'malah', 'mlh': 'malah', 'leg': 'lag', 'ngelek': 'lag',
    'striming': 'streaming', 'tivi': 'tv', 'langgan': 'langganan',
    'dpt': 'dapat', 'pke': 'pake', 'trnyata': 'ternyata',
})

def norm(x):
    x = str(x).lower()
    x = re.sub(r'http\S+|www\.\S+', ' ', x)
    x = re.sub(r'[^a-z0-9\s]', ' ', x)
    x = re.sub(r'([a-z])\1{2,}', r'\1', x)
    x = re.sub(r'\b([a-z]+)2\b', r'\1 \1', x)   # muter2 -> muter muter
    return ' '.join(SLANG.get(w, w) for w in x.split())

r['content_clean'] = r['content_raw'].map(norm)
c = r['content_clean']

FAIL += ['tidak masuk', 'kadaluarsa']
PEMBAYARAN += ['kadaluarsa', 'kedaluwarsa', 'expired',
               'langganan sendiri', 'berlangganan sendiri', 'bundling']
PAYWALL += ['sekarang bayar', 'naik harga', 'harganya naik',
            'pajak', 'harus beli']
TV += ['scan', 'barcode', 'qr', 'proyektor', 'tv',
       'disambungin', 'disambungkan', 'sambungkan']
LAG += ['sinyal']
ERROR += ['kebuka sendiri', 'terbuka sendiri']
KONTEN += ['hak siar', 'jadwal']

TOPICS = {
    'pembayaran': kw(PEMBAYARAN),
    'akses_setelah_bayar': kw(PAY_TERM) & kw(FAIL),
    'login_akun': kw(LOGIN),
    'error_crash': kw(ERROR),
    'gagal_putar': kw(GAGAL_PUTAR) | (kw(['tidak bisa']) &
                   kw(['nonton', 'menonton', 'live', 'streaming',
                       'lihat', 'dipakai'])),
    'lag_buffering': kw(LAG),
    'iklan': c.str.contains(r'\biklan\w*', regex=True),
    'konten': kw(KONTEN),
    'paywall_harga': kw(PAYWALL),
    'tv_device': kw(TV),
}
for k, m in TOPICS.items():
    r['is_' + k] = m
r['mentions_sport'] = kw(SPORT + ['euro', 'europa'])
r['n_topics'] = r[flag_cols].sum(axis=1)
r['n_words'] = c.str.split().str.len()

r['complaint_topic'] = 'lainnya'
for k in reversed(list(TOPICS)):
    r.loc[r['is_' + k], 'complaint_topic'] = k
print('patched:', r.shape)

patched: (46649, 37)


In [ ]:
p = '/content/drive/MyDrive/mnc-analytics/final/complaint_validation_sample.csv'
try:
    s = pd.read_csv(p, sep=';', encoding='utf-8-sig')
except UnicodeDecodeError:
    s = pd.read_csv(p, sep=';', encoding='cp1252')
s['complaint_manual'] = s['complaint_manual'].astype(str).str.strip()
print('labeled:', (s['complaint_manual'] != 'nan').sum(), '/ 100')

# exact match on primary topic
s['correct'] = s['complaint_topic'] == s['complaint_manual']

# lenient: manual label is among ANY flagged topic
flags = r.set_index('review_id')[flag_cols]
def in_flags(row):
    if row['complaint_manual'] == 'lainnya':
        return row['complaint_topic'] == 'lainnya'
    col = 'is_' + row['complaint_manual']
    return col in flags.columns and bool(flags.at[row['review_id'], col])
s['in_flags'] = s.apply(in_flags, axis=1)

print(f"primary-topic accuracy: {s['correct'].mean():.0%}")
print(f"manual topic among flags: {s['in_flags'].mean():.0%}")
print('\nconfusion (rows = auto, cols = manual):')
print(pd.crosstab(s['complaint_topic'], s['complaint_manual']).to_string())
print('\nmismatches:')
print(s.loc[~s['correct'], ['app', 'content_clean', 'complaint_topic',
                            'complaint_manual']].to_string())

labeled: 100 / 100
primary-topic accuracy: 68%
manual topic among flags: 75%

confusion (rows = auto, cols = manual):
complaint_manual     akses_setelah_bayar  error_crash  gagal_putar  iklan  konten  lag_buffering  lainnya  login_akun  paywall_harga  pembayaran
complaint_topic                                                                                                                                 
akses_setelah_bayar                   14            0            0      0       0              0        0           0              2           0
error_crash                            2            3            1      1       0              1        0           0              0           0
gagal_putar                            0            0            2      0       2              0        0           0              1           0
iklan                                  0            0            0      7       0              0        0           0              0           0
konten      

In [ ]:
SLANG.update({
    'aktv': 'aktif', 'aktiv': 'aktif', 'redem': 'redeem',
    'regis': 'registrasi', 'sndri': 'sendiri', 'uda': 'sudah',
    'lodinh': 'loading', 'buffring': 'buffering',
    'gaada': 'tidak ada', 'gada': 'tidak ada',
    'berlanganan': 'berlangganan', 'metoda': 'metode',
})
r['content_clean'] = r['content_raw'].map(norm)
c = r['content_clean']

def kw(words):
    pat = r'\b(?:' + '|'.join(words) + r')(?:nya|an|kan|in)?\b'
    return c.str.contains(pat, regex=True)

def drop(lst, bad):
    return [w for w in lst if w not in bad]

PAY2 = PAY_TERM + ['aktivasi', 'redeem', 'pembayaran']
FAIL2 = drop(FAIL, {'suruh', 'disuruh'})
FAIL2 += ['error', 'tidak otomatis masuk']
FAIL2 += ['tidak masuk', 'belum masuk']
print('A ok')

A ok


In [ ]:
PEMB2 = drop(PEMBAYARAN, {'otomatis'})
PEMB2 += ['perpanjang otomatis', 'otomatis perpanjang']
PEMB2 += ['langganan otomatis', 'auto renew', 'auto debit']
PEMB2 += ['perpanjangan', 'kesedot', 'tersedot', 'balikin']
PEMB2 += ['dikembalikan', 'gagal daftar paket']
PEMB2 += ['daftar paket gagal', 'pembayaran gagal']

PAYW2 = drop(PAYWALL, {'gratis'})
PAYW2 += ['nonton gratis', 'gratiskan', 'katanya gratis']
PAYW2 += ['bukan gratis', 'suruh beli', 'disuruh beli']
PAYW2 += ['suruh upgrade', 'harus upgrade', 'upgrade premium']
PAYW2 += ['ujung ujungnya duit', 'pilihan paket']
PAYW2 += ['paket sendiri', 'top up']

LOGIN2 = drop(LOGIN, {'daftar'})
LOGIN2 += ['tautan', 'tautkan', 'ditautkan', 'kvision']
LOGIN2 += ['k vision', 'sambung ke vision', 'sambungkan akun']
LOGIN2 += ['hubungkan akun', 'gagal daftar', 'tidak bisa daftar']
LOGIN2 += ['kode verifikasi', 'verifi']
print('B ok')

B ok


In [ ]:
BAD_ERR = {'update', 'diupdate', 'install', 'uninstall',
           'penyimpanan', 'memori'}
ERR2 = drop(ERROR, BAD_ERR)
ERR2 += ['tidak bisa update', 'tidak bisa di update']
ERR2 += ['tidak bisa diupdate', 'gagal update']
ERR2 += ['setelah update', 'habis update', 'sesudah update']
ERR2 += ['tidak bisa install', 'gagal install']
ERR2 += ['tidak bisa digunakan', 'tidak bisa digunain', 'retry']

LAG2 = LAG + ['patah', 'ketinggalan', 'telat', 'nunggu lama']
LAG2 += ['leg', 'lelet']

KON2 = KONTEN + ['kualitas', 'season', 'uptodate', 'up to date']
KON2 += ['dimasukin', 'dimasukkan', 'anime', 'drakor']

WATCH = ['nonton', 'menonton', 'live', 'streaming', 'lihat', 'dipakai']
TOPICS = {
    'akses_setelah_bayar': kw(PAY2) & kw(FAIL2),
    'pembayaran': kw(PEMB2),
    'paywall_harga': kw(PAYW2),
    'login_akun': kw(LOGIN2),
    'iklan': c.str.contains(r'\biklan\w*', regex=True),
    'konten': kw(KON2),
    'error_crash': kw(ERR2),
    'lag_buffering': kw(LAG2),
    'gagal_putar': kw(GAGAL_PUTAR) | (kw(['tidak bisa']) & kw(WATCH)),
    'tv_device': kw(TV),
}
print('C ok')

C ok


In [ ]:
for k, m in TOPICS.items():
    r['is_' + k] = m
flag_cols = ['is_' + k for k in TOPICS]
r['n_topics'] = r[flag_cols].sum(axis=1)
r['complaint_topic'] = 'lainnya'
for k in reversed(list(TOPICS)):
    r.loc[r['is_' + k], 'complaint_topic'] = k

new = r.set_index('review_id')['complaint_topic']
s['complaint_topic_v3'] = s['review_id'].map(new)
acc = (s['complaint_topic_v3'] == s['complaint_manual']).mean()
print(f'same-sample accuracy (optimistic): {acc:.0%}')

neg = r[r['score'] <= 2]
un = neg['complaint_topic'].eq('lainnya').groupby(neg['app']).mean()
print('\nuntagged negative (%):')
print((un * 100).round(1))

pool = neg[~neg['review_id'].isin(s['review_id'])]
cols = ['review_id', 'app', 'score', 'content_clean', 'complaint_topic']
fresh = pool.sample(50, random_state=7)[cols].copy()
fresh['complaint_manual'] = ''
fresh.to_csv(BASE + 'final/complaint_validation_fresh50.csv',
             index=False, sep=';', encoding='utf-8-sig')
r.to_csv(BASE + 'clean/app_reviews_clean.csv', index=False)
print('\nsaved')

same-sample accuracy (optimistic): 85%

untagged negative (%):
app
RCTIplus      36.1
Vidio         21.3
Visionplus    22.8
Name: complaint_topic, dtype: float64

saved


In [ ]:
p = BASE + 'final/complaint_validation_fresh50.csv'
try:
    f = pd.read_csv(p, sep=';', encoding='utf-8-sig')
except UnicodeDecodeError:
    f = pd.read_csv(p, sep=';', encoding='cp1252')
f['complaint_manual'] = f['complaint_manual'].astype(str).str.strip()
print('labeled:', (f['complaint_manual'] != 'nan').sum(), '/ 50')

f['correct'] = f['complaint_topic'] == f['complaint_manual']
print(f"fresh-samp
le accuracy: {f['correct'].mean():.0%}")

print('\nconfusion (rows = auto, cols = manual):')
print(pd.crosstab(f['complaint_topic'], f['complaint_manual'])
      .to_string())
print('\nmismatches:')
cols = ['app', 'content_clean', 'complaint_topic', 'complaint_manual']
print(f.loc[~f['correct'], cols].to_string())

labeled: 50 / 50
fresh-sample accuracy: 82%

confusion (rows = auto, cols = manual):
complaint_manual     akses_setelah_bayar  error_crash  iklan  konten  lag_buffering  lainnya  login_akun  paywall_harga  pembayaran
complaint_topic                                                                                                                    
akses_setelah_bayar                   14            0      0       0              0        0           1              1           1
error_crash                            0            4      0       0              0        0           0              0           0
iklan                                  0            0      5       0              0        0           0              0           0
konten                                 0            0      0       2              2        0           0              0           0
lag_buffering                          0            0      0       0              6        0           0              0    